# Combined Learning Notebook — FedAvg + proposed extensions

Full walk in topological order: foundations are linked out (see `learning-map/README.md`), then paper concepts, then improvements.

# Finite-Sum Objective
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `01-finite-sum-objective`
**Graph:** `paper`
**Prerequisites:** [expectation](https://github.com/pleyva2004/math-foundations/blob/main/concepts/expectation.md), [empirical risk](https://github.com/pleyva2004/math-foundations/blob/main/concepts/empirical-risk.md)
**Used by:** downstream nodes

## Plain-English intro
Almost every supervised-learning goal is to make the *average* loss over a fixed training set as small as possible. FedAvg writes this goal as a **finite sum**: one term per training example, divided by the number of examples. This single average is the global objective; every later idea (client splits, local updates, averaging) is just a way of taking it apart and minimizing it across machines.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w\in\mathbb{R}^d$ | "w in R-d" | Model parameter vector ($d$ parameters) <!-- TODO add to foundations --> |
| $f(w)$ | "f of w" | Global objective: mean loss over all $n$ examples <!-- TODO add to foundations --> |
| $f_i(w)=\ell(x_i,y_i;w)$ | "f-sub-i of w" | Per-example loss on $(x_i,y_i)$ <!-- TODO add to foundations --> |
| $n$ | "n" | Total number of training examples <!-- TODO add to foundations --> |
| $\ell(\cdot)$ | "ell" | Loss function comparing prediction to label <!-- TODO add to foundations --> |

## Formal definition
$$
\min_{w\in\mathbb{R}^d} f(w),\qquad
f(w)\ \stackrel{\text{def}}{=}\ \frac{1}{n}\sum_{i=1}^{n} f_i(w),
\qquad f_i(w)=\ell(x_i,y_i;w). \tag{0.1}
$$
No convexity or smoothness of $\ell$ is assumed; $f$ is in general non-convex.

## Why this matters
This is the global objective FedAvg minimizes; everything else decomposes it — e.g. the partition identity $f(w)=\sum_k \frac{n_k}{n}F_k(w)$ in §1 / Eq. (1.1) of `02-math-deep-dive.md` rests directly on Eq. (0.1).

## Code
The aligned runnable demo lives at [`../code/01-finite-sum-objective.py`](../code/01-finite-sum-objective.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Finite-Sum Objective: f(w) = (1/n) * sum_{i=1}^n f_i(w)
n = 12 toy 1-D regression examples; f_i = squared error.
------------------------------------------------------------
```

## Try it yourself
- Exercise 1: Replace the squared-error $f_i$ with absolute error $|w x_i - y_i|$ and confirm $f$ is still exactly the mean of the $f_i$.
- Exercise 2: Add a second parameter (intercept $b$) so $w\in\mathbb{R}^2$, and check the grid-minimizer recovers both the true slope and intercept.

## Further reading
- McMahan et al., "Communication-Efficient Learning of Deep Networks from Decentralized Data," AISTATS 2017, Eq. (1). https://arxiv.org/abs/1602.05629


In [ ]:
"""Finite-Sum Objective -- concept 01-finite-sum-objective of the paper learning map.

Witness that the global objective f(w) is literally the average of the per-example
losses f_i(w), i.e. f(w) = (1/n) sum_i f_i(w) (Eq. 0.1 of the deep dive).
Runnable code analog of concepts/01-finite-sum-objective.py.
"""
import numpy as np


def main():
    np.random.seed(0)

    # ---- n=12 toy 1-D regression examples (x_i, y_i) ----
    n = 12
    xs = np.linspace(-2.0, 2.0, n)          # inputs x_i
    true_w = 1.7                            # ground-truth slope (intercept-free)
    noise = 0.3 * np.random.randn(n)        # small label noise
    ys = true_w * xs + noise                # labels y_i

    # Per-example squared-error loss f_i(w) = (w*x_i - y_i)^2 = l(x_i, y_i; w).
    def f_i(w, i):
        return (w * xs[i] - ys[i]) ** 2

    # Global objective f(w) = (1/n) sum_i f_i(w).
    def f(w):
        return float(np.mean([f_i(w, i) for i in range(n)]))

    print("Finite-Sum Objective: f(w) = (1/n) * sum_{i=1}^n f_i(w)")
    print("n = %d toy 1-D regression examples; f_i = squared error." % n)
    print("-" * 60)

    for w in (0.0, 1.7):
        per_example = np.array([f_i(w, i) for i in range(n)])
        f_val = f(w)                                  # via the f() helper
        manual_mean = float(per_example.mean())       # explicit average
        manual_sum = float(per_example.sum() / n)     # (1/n)*sum form

        print("w = %.3f" % w)
        print("  per-example losses f_i(w):")
        print("   ", np.array2string(per_example, precision=4,
                                     floatmode="fixed", separator=", "))
        print("  f(w) via mean of f_i      = %.6f" % f_val)
        print("  f(w) via (1/n)*sum f_i    = %.6f" % manual_sum)
        print("  numpy .mean() of f_i      = %.6f" % manual_mean)
        # f is LITERALLY the average: all three agree to machine precision.
        assert abs(f_val - manual_mean) < 1e-12
        assert abs(f_val - manual_sum) < 1e-12
        print("  -> f(w) IS the average of the f_i(w)  [verified, diff < 1e-12]")
        print("-" * 60)

    # Sanity: the minimizer of this finite sum sits near true_w (least squares).
    grid = np.linspace(0.0, 3.0, 3001)
    losses = np.array([f(w) for w in grid])
    w_star = grid[int(np.argmin(losses))]
    print("argmin_w f(w) on grid = %.3f  (true slope = %.2f)" % (w_star, true_w))


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Client-Partition Decomposition
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `02-client-partition-decomposition`
**Graph:** `paper`
**Prerequisites:** [01-finite-sum-objective](01-finite-sum-objective.md), [partition of a set](https://github.com/pleyva2004/math-foundations/blob/main/concepts/partition-of-a-set.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg distributes the $n$ training examples across $K$ clients, where client $k$ holds an index set $P_k$. Each client has its own *local* objective $F_k$, the average loss over only its examples. This concept shows the global objective $f$ is exactly the size-weighted average of the local objectives: nothing is approximated, and no assumption is made about *how* the data was split.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w\in\mathbb{R}^d$ | "w in R-d" | Model parameter vector <!-- TODO add to foundations --> |
| $f(w)$ | "f of w" | Global objective: mean loss over all $n$ examples <!-- TODO add to foundations --> |
| $f_i(w)$ | "f-sub-i of w" | Per-example loss on example $i$ <!-- TODO add to foundations --> |
| $n$ | "n" | Total number of training examples <!-- TODO add to foundations --> |
| $K$ | "K" | Total number of clients <!-- TODO add to foundations --> |
| $P_k$ | "P-sub-k" | Index set of examples held by client $k$ <!-- TODO add to foundations --> |
| $n_k=\lvert P_k\rvert$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $F_k(w)$ | "cap-F-sub-k of w" | Client $k$'s local objective <!-- TODO add to foundations --> |

## Formal definition
$$
\bigcup_{k=1}^{K}P_k=[n],\quad P_k\cap P_{k'}=\varnothing\ (k\ne k'),\quad n_k:=|P_k|,\ \textstyle\sum_k n_k=n;
$$
$$
F_k(w):=\frac{1}{n_k}\sum_{i\in P_k}f_i(w)\quad\Longrightarrow\quad f(w)=\sum_{k=1}^{K}\frac{n_k}{n}\,F_k(w)\quad\forall w\in\mathbb{R}^d.
$$
The weight $\frac{n_k}{n}\cdot\frac{1}{n_k}=\frac1n$ collapses, re-indexing the partitioned sum back to $\frac1n\sum_{i=1}^n f_i$.

## Why this matters
This is the structural backbone of FedAvg, appearing as Eq. (1.1) of `02-math-deep-dive.md`. It is an exact algebraic identity for *any* partition, even adversarial/non-IID, which is why the IID assumption is needed only for *quality*, never for *correctness*.

## Code
The aligned runnable demo lives at [`../code/02-client-partition-decomposition.py`](../code/02-client-partition-decomposition.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.
**Expected output preview:**
```
Client-Partition Decomposition: f(w) = sum_k (n_k/n) F_k(w)  [FedAvg Eq. 1.1]
n=60 examples, K=5 clients, sizes n_k=[7, 25, 15, 1, 12] (unequal), n_k/n weights sum=1.0
max |residual| over 6 random w = 8.88e-16  (~machine epsilon => identity holds)
```

## Try it yourself
- Exercise 1: Make the partition NOT cover all of $[n]$ (drop one index). Confirm the residual is no longer ~1e-15, showing coverage is load-bearing.
- Exercise 2: Replace `n_k/n` with uniform client weights `1/K` and observe the residual blow up unless all $n_k$ are equal.

## Further reading
- McMahan et al. (2017), "Communication-Efficient Learning of Deep Networks from Decentralized Data," arXiv:1602.05629, Eq. (1).


In [ ]:
"""Client-Partition Decomposition -- concept 02-client-partition-decomposition of the paper learning map.

Witnesses FedAvg's exact identity f(w) = sum_k (n_k/n) F_k(w) (Eq. 1.1): the global
finite-sum objective is the size-weighted mixture of per-client local objectives, for
ANY partition of the n examples into K clients (no IID assumption needed).
Runnable code analog of concepts/02-client-partition-decomposition.py.
"""
import numpy as np


def make_partition(n, K, rng):
    """Cut [n] into K disjoint, nonempty, UNEQUAL index blocks P_1..P_K covering all of [n]."""
    cuts = np.sort(rng.choice(np.arange(1, n), size=K - 1, replace=False))
    bounds = np.concatenate(([0], cuts, [n]))
    perm = rng.permutation(n)  # shuffle so blocks are not just contiguous originals
    return [perm[bounds[k]:bounds[k + 1]] for k in range(K)]


def main():
    rng = np.random.default_rng(0)
    np.random.seed(0)

    n, K, d = 60, 5, 4
    # n toy "examples": each f_i(w) is a smooth per-example loss 0.5 * ||A_i w - b_i||^2.
    A = rng.standard_normal((n, d))
    b = rng.standard_normal(n)

    def f_i(w, i):
        r = A[i] @ w - b[i]
        return 0.5 * r * r

    def f(w):  # global objective: mean over all n examples (Eq. 0.1)
        return np.mean([f_i(w, i) for i in range(n)])

    P = make_partition(n, K, rng)
    n_k = np.array([len(Pk) for Pk in P])
    assert n_k.sum() == n and (n_k >= 1).all()              # coverage + nonempty (P)
    assert len(np.unique(np.concatenate(P))) == n           # disjointness (P)

    def F_k(w, k):  # local objective: mean over client k's examples (Eq. 0.2)
        return np.mean([f_i(w, i) for i in P[k]])

    print("Client-Partition Decomposition: f(w) = sum_k (n_k/n) F_k(w)  [FedAvg Eq. 1.1]")
    print("n=%d examples, K=%d clients, sizes n_k=%s (unequal), n_k/n weights sum=%.1f"
          % (n, K, [int(x) for x in n_k], (n_k / n).sum()))
    print("Verifying the exact algebraic identity at several random w:")

    max_residual = 0.0
    for trial in range(6):
        w = rng.standard_normal(d)
        lhs = f(w)
        rhs = sum((n_k[k] / n) * F_k(w, k) for k in range(K))
        res = abs(lhs - rhs)
        max_residual = max(max_residual, res)
        print("  w#%d: f(w)=%+.10f   sum_k (n_k/n)F_k(w)=%+.10f   |residual|=%.2e"
              % (trial, lhs, rhs, res))

    print("max |residual| over 6 random w = %.2e  (~machine epsilon => identity holds)"
          % max_residual)
    assert max_residual < 1e-12, "decomposition identity failed"
    print("WITNESS PASS: the size-weighted mixture reconstructs f exactly, for this partition.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# IID vs Non-IID
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `03-iid-noniid-dichotomy`
**Graph:** `paper`
**Prerequisites:** [02-client-partition-decomposition](02-client-partition-decomposition.md), [unbiased estimator](https://github.com/pleyva2004/math-foundations/blob/main/concepts/unbiased-estimator.md)
**Used by:** downstream nodes

## Plain-English intro
The decomposition $f=\sum_k\frac{n_k}{n}F_k$ holds for *any* split of the data. The IID/non-IID dichotomy asks a *statistical* question on top of it: does a single client's local objective $F_k$ resemble the global $f$? If the data is sprinkled across clients uniformly at random (IID), each $F_k$ is an unbiased proxy for $f$. If instead each client hoards one kind of example (non-IID), $F_k$ can be an arbitrarily bad approximation to $f$ — yet the size-weighted mixture of all $F_k$ still equals $f$ exactly.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w$ | "w" | Model parameter vector being evaluated <!-- TODO add to foundations --> |
| $f(w)$ | "f of w" | Global objective: mean loss over all $n$ examples <!-- TODO add to foundations --> |
| $F_k(w)$ | "cap-F-sub-k of w" | Client $k$'s local objective (mean loss on $P_k$) <!-- TODO add to foundations --> |
| $P_k$ | "P-sub-k" | Index set of examples held by client $k$ <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $\mathbb{E}_{P_k}[\cdot]$ | "expectation over P-k" | Average over the random draw of $P_k$ <!-- TODO add to foundations --> |

## Formal definition
$$
\textbf{IID:}\quad P_k\ \text{a uniform-random partition}\ \Longrightarrow\ \mathbb{E}_{P_k}\!\big[F_k(w)\big]=f(w)\quad\forall w\ \text{(Eq. 1.2, unbiased proxy)}.
$$
$$
\textbf{Non-IID:}\ \text{the negation —}\ \exists\,w:\ \mathbb{E}_{P_k}\!\big[F_k(w)\big]\neq f(w),\ \text{equivalently}\ \sup_w\big|F_k(w)-f(w)\big|\ \text{unbounded.}
$$
The exact identity $f(w)=\sum_{k}\tfrac{n_k}{n}F_k(w)$ is **unaffected** in either regime; only per-client statistical fidelity changes.

## Why this matters
This is the regime that distinguishes federated optimization from ordinary distributed SGD; it appears as Eq. (1.2) and the dichotomy table of §1 in `02-math-deep-dive.md`, and drives the FedAvg non-IID robustness questions that motivate `05-improvements.tex` M.1 (heterogeneity drift $\Delta=\eta^2\mathrm{Cov}_k(H_k,g_k)$).

## Code
The aligned runnable demo lives at [`../code/03-iid-noniid-dichotomy.py`](../code/03-iid-noniid-dichotomy.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Fixed point w; global objective f(w) = 8.2501
IID partition (uniform-random shards): mean |F_k(w) - f(w)| = 0.2153   <- unbiased, E[F_k]~f  (Eq. 1.2)
Non-IID partition (sort-by-label shards): mean |F_k(w) - f(w)| = 6.4000   <- arbitrarily bad proxy
```

## Try it yourself
- Exercise 1: Increase shard count $K$ (fewer examples per shard). Does the IID gap grow? Relate it to the $1/\sqrt{n_k}$ shrinkage of an unbiased estimator's standard error.
- Exercise 2: Make the non-IID split *two* labels per shard instead of one. Confirm the gap shrinks but the mixture still reconstructs $f(w)$ exactly.

## Further reading
- McMahan et al. 2017, *Communication-Efficient Learning of Deep Networks from Decentralized Data* (arXiv:1602.05629), §3 pathological non-IID MNIST.
- `02-math-deep-dive.md` §1 (Eq. 1.1, 1.2 and the IID/non-IID status table).


In [ ]:
"""IID vs Non-IID -- concept 03-iid-noniid-dichotomy of the paper learning map.

Monte-Carlo witness: a uniform-random (IID) partition makes each local objective
F_k(w) an unbiased proxy for the global f(w), so E[F_k]~f; a sort-by-label
(non-IID) partition makes F_k an arbitrarily bad approximation to f at a fixed w.
Runnable code analog of concepts/03-iid-noniid-dichotomy.py.
"""
import numpy as np


def build_data(n=6000, num_classes=10, d=5):
    """Labelled (x, y); per-example loss f_i(w) is squared-error to label y_i."""
    rng = np.random.default_rng(0)
    y = np.repeat(np.arange(num_classes), n // num_classes)  # balanced labels
    # one deterministic feature vector per class so f_i depends on the label
    centers = rng.standard_normal((num_classes, d))
    x = centers[y] + 0.01 * rng.standard_normal((n, d))
    return np.ascontiguousarray(x), y, np.ascontiguousarray(centers)


def f_i(w, x, y, centers):
    """Per-example loss: how badly w 'scores' example i relative to its class."""
    # score = x . w ; target = class-center projection (dot via elementwise sum
    # to avoid spurious numpy-2.0 matmul FPE warnings on (n,d)@(d,) shapes)
    score = (x * w).sum(axis=1)
    target = (centers[y] * w).sum(axis=1)
    return (score - target) ** 2 + (y - 4.5) ** 2  # label-dependent term


def F(w, idx, x, y, centers):
    """Average loss over an index set (F_k if idx=P_k, f if idx=all)."""
    return float(np.mean(f_i(w, x[idx], y[idx], centers)))


def main():
    np.random.seed(0)
    rng = np.random.default_rng(0)
    n, K = 6000, 10
    x, y, centers = build_data(n=n)
    all_idx = np.arange(n)
    w = rng.standard_normal(x.shape[1])         # one fixed evaluation point w
    f_w = F(w, all_idx, x, y, centers)          # global objective f(w)

    # IID: uniform-random partition into K equal shards (each shard mixes labels)
    perm = rng.permutation(n)
    iid_parts = np.array_split(perm, K)
    iid_gap = np.mean([abs(F(w, p, x, y, centers) - f_w) for p in iid_parts])

    # Non-IID: sort by label, then split -> each shard holds ~1 label
    sorted_idx = all_idx[np.argsort(y, kind="stable")]
    noniid_parts = np.array_split(sorted_idx, K)
    noniid_gap = np.mean([abs(F(w, p, x, y, centers) - f_w) for p in noniid_parts])

    print("Fixed point w; global objective f(w) = {:.4f}".format(f_w))
    print("IID partition (uniform-random shards): mean |F_k(w) - f(w)| = "
          "{:.4f}   <- unbiased, E[F_k]~f  (Eq. 1.2)".format(iid_gap))
    print("Non-IID partition (sort-by-label shards): mean |F_k(w) - f(w)| = "
          "{:.4f}   <- arbitrarily bad proxy".format(noniid_gap))
    print("Heterogeneity blow-up factor (non-IID / IID) = {:.1f}x".format(
        noniid_gap / iid_gap))
    # Identity (1.1) still holds exactly under BOTH partitions: weighted mix = f
    for name, parts in (("IID", iid_parts), ("non-IID", noniid_parts)):
        mix = sum((len(p) / n) * F(w, p, x, y, centers) for p in parts)
        print("  size-weighted mix of F_k under {:>7s} = {:.4f}  (== f(w), "
              "identity 1.1 unbroken)".format(name, mix))


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# FedSGD as Exact Gradient Descent
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `04-fedsgd-gradient-descent`
**Graph:** `paper`
**Prerequisites:** [02-client-partition-decomposition](02-client-partition-decomposition.md), [gradient](https://github.com/pleyva2004/math-foundations/blob/main/concepts/gradient.md), [gradient descent](https://github.com/pleyva2004/math-foundations/blob/main/concepts/gradient-descent.md)
**Used by:** downstream nodes

## Plain-English intro
FedSGD has every client compute the average gradient of its own data at the shared model $w_t$, then the server combines them. When *all* clients participate ($C{=}1$), the size-weighted sum of those local gradients equals the gradient of the global objective exactly. So one FedSGD round is nothing exotic: it is textbook deterministic full-batch gradient descent on $f$, with the clients merely evaluating disjoint partial sums of one global gradient. This makes FedSGD the baseline endpoint against which all of FedAvg is measured.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w_t$ | "w-sub-t" | Shared model parameters at round $t$ <!-- TODO add to foundations --> |
| $f(w)$ | "f of w" | Global objective: mean loss over all $n$ examples <!-- TODO add to foundations --> |
| $F_k(w)$ | "cap-F-sub-k of w" | Client $k$'s local objective (mean loss on $P_k$) <!-- TODO add to foundations --> |
| $g_k=\nabla F_k(w_t)$ | "g-sub-k" | Client $k$'s local average gradient at $w_t$ <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $n$ | "n" | Total number of training examples ($n=\sum_k n_k$) <!-- TODO add to foundations --> |
| $\eta$ | "eta" | Learning rate <!-- TODO add to foundations --> |
| $C$ | "C" | Fraction of clients selected per round ($C{=}1$ here) <!-- TODO add to foundations --> |

## Formal definition
$$
g_k \stackrel{\text{def}}{=} \nabla F_k(w_t),\qquad
\sum_{k=1}^{K}\frac{n_k}{n}\,g_k \;=\; \nabla f(w_t)\quad\text{(Eq. 2.2)};
$$
$$
\text{the }C{=}1\text{ FedSGD update}\quad
w_{t+1}=w_t-\eta\sum_{k=1}^{K}\frac{n_k}{n}g_k \;=\; w_t-\eta\,\nabla f(w_t)\quad\text{(Eq. 2.3)}.
$$
The identity is exact for *any* partition (no IID assumption); it follows by differentiating $f=\sum_k\frac{n_k}{n}F_k$ term-by-term.

## Why this matters
Identifies FedSGD as exact distributed full-batch gradient descent -- the baseline endpoint of the $(C,E,B)$ family. Appears as Eq. (2.2)-(2.3) of `02-math-deep-dive.md` (§2).

## Code
The aligned runnable demo lives at [`../code/04-fedsgd-gradient-descent.py`](../code/04-fedsgd-gradient-descent.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
FedSGD (C=1) gradient-aggregation identity:  sum_k (n_k/n) g_k  ==  grad f(w_t)
client sizes n_k = [2, 3, 3, 4], total n = 12, weights n_k/n = [0.1667, 0.25, 0.25, 0.3333]
residual ||sum_k (n_k/n) g_k - grad f(w_t)|| = 5.088e-16  (expect ~1e-12)
```

## Try it yourself
- Exercise 1: Make the partition unbalanced (e.g. one client holds 9 of 12 examples). Confirm the weighted residual stays ~1e-16 but the *unweighted* mean $\frac1K\sum_k g_k$ no longer matches $\nabla f$.
- Exercise 2: Set $C<1$ by aggregating over a random client subset $S_t$ with weights $n_k/m_t$. Observe that the single-round residual is now nonzero -- FedSGD becomes a stochastic estimator of $\nabla f$, not an equality (see §5).

## Further reading
- McMahan et al., "Communication-Efficient Learning of Deep Networks from Decentralized Data," AISTATS 2017, §2 (arXiv:1602.05629).


In [ ]:
"""FedSGD as Exact Gradient Descent -- concept 04-fedsgd-gradient-descent of the paper learning map.

At C=1, FedSGD aggregates per-client gradients g_k=grad F_k(w_t) with weights n_k/n;
this reconstructs the full-data gradient grad f(w_t) exactly, so the update is plain
deterministic full-batch gradient descent on f.
Runnable code analog of concepts/04-fedsgd-gradient-descent.py.
"""

import numpy as np


def main():
    np.random.seed(0)

    # Toy convex problem: f_i(w) = 0.5 * (a_i . w - b_i)^2, a least-squares finite sum.
    # Global objective f(w) = (1/n) sum_i f_i(w). We pick a partition of the n examples
    # into K clients and verify the gradient-aggregation identity (Eq. 2.2).
    n, d, K = 12, 3, 4
    A = np.random.randn(n, d)
    b = np.random.randn(n)
    w_t = np.random.randn(d)  # shared model broadcast to every client

    # Per-example gradient: grad f_i(w) = (a_i . w - b_i) * a_i
    def grad_f_i(idx, w):
        return (A[idx] @ w - b[idx]) * A[idx]

    # Partition [n] into K disjoint, exhaustive index sets of unequal sizes (Eq. P).
    perm = np.random.permutation(n)
    P = [perm[0:2], perm[2:5], perm[5:8], perm[8:12]]  # sizes 2,3,3,4 -> sum = 12 = n
    n_k = np.array([len(p) for p in P])
    assert n_k.sum() == n and len(np.unique(np.concatenate(P))) == n

    # Full-data gradient grad f(w_t) = (1/n) sum_i grad f_i(w_t)  (the ground truth).
    grad_f = np.mean([grad_f_i(i, w_t) for i in range(n)], axis=0)

    # Each client computes its LOCAL average gradient g_k = grad F_k(w_t)  (Eq. 2.1).
    g = [np.mean([grad_f_i(i, w_t) for i in P[k]], axis=0) for k in range(K)]

    # FedSGD (C=1) aggregation: weighted sum with weights n_k/n  (Eq. 2.2).
    agg = sum((n_k[k] / n) * g[k] for k in range(K))

    residual = np.linalg.norm(agg - grad_f)

    # The C=1 FedSGD step equals the centralized full-batch GD step (Eq. 2.3).
    eta = 0.1
    w_fedsgd = w_t - eta * agg
    w_gd = w_t - eta * grad_f
    step_residual = np.linalg.norm(w_fedsgd - w_gd)

    print("FedSGD (C=1) gradient-aggregation identity:  sum_k (n_k/n) g_k  ==  grad f(w_t)")
    print("client sizes n_k = {}, total n = {}, weights n_k/n = {}".format(
        n_k.tolist(), n, np.round(n_k / n, 4).tolist()))
    print("residual ||sum_k (n_k/n) g_k - grad f(w_t)|| = {:.3e}  (expect ~1e-12)".format(residual))
    print("step  ||(w_t - eta*agg) - (w_t - eta*grad f)|| = {:.3e}  -> FedSGD step == full-batch GD step".format(
        step_residual))

    assert residual < 1e-12, "aggregation identity must hold to ~1e-12"
    assert step_residual < 1e-12, "FedSGD update must equal centralized GD update"
    print("PASS: C=1 FedSGD is exact distributed full-batch gradient descent.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Gradient- vs Model-Averaging Equivalence
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `05-gradient-model-averaging-equivalence`
**Graph:** `paper`
**Prerequisites:** [04-fedsgd-gradient-descent](04-fedsgd-gradient-descent.md)
**Used by:** downstream nodes

## Plain-English intro
FedSGD can be run two ways that give the *same* answer for one local step: the server can average everyone's gradients and then take a step, OR each client can take its own step and the server averages the resulting models. They coincide because all clients start from the *same* point $w_t$ and the weights $n_k/n$ sum to 1. Take a *second* local step and the two schemes no longer agree — that gap is exactly what FedAvg exploits.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w_t$ | "w-sub-t" | Shared model at round $t$ (common start for every client) <!-- TODO add to foundations --> |
| $w^k$ | "w-super-k" | Client $k$'s model after its local step(s) <!-- TODO add to foundations --> |
| $g_k=\nabla F_k(w_t)$ | "g-sub-k" | Client $k$'s local gradient at $w_t$ <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $n=\sum_k n_k$ | "n" | Total examples; makes $n_k/n$ a weight summing to 1 <!-- TODO add to foundations --> |
| $\eta$ | "eta" | Learning rate <!-- TODO add to foundations --> |

## Formal definition
$$
\sum_{k}\frac{n_k}{n}\,w^k=\sum_{k}\frac{n_k}{n}\bigl(w_t-\eta g_k\bigr)=\underbrace{\Bigl(\sum_k\tfrac{n_k}{n}\Bigr)}_{=\,1}w_t-\eta\sum_k\frac{n_k}{n}g_k=w_t-\eta\sum_k\frac{n_k}{n}g_k.
$$
For $\tau\ge2$ local steps the composite local map is nonlinear in $w_t$, so $\sum_k\frac{n_k}{n}w^k_{(\tau)}\neq$ the centralized averaged-gradient iterate (Eq. 3.4).

## Why this matters
This one-step identity is Eq. (3.2) of `02-math-deep-dive.md` (§3): the hinge that lets FedSGD be reorganized as model averaging, which FedAvg then generalizes by iterating the local step.

## Code
The aligned runnable demo lives at [`../code/05-gradient-model-averaging-equivalence.py`](../code/05-gradient-model-averaging-equivalence.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Concept 05: model-average of one local step == gradient-average step (Eq. 3.2)
ONE local step:
  ||model-average - gradient-average|| = 1.110e-16  (should be ~1e-15)
```

## Try it yourself
- Exercise 1: Change the client weights so they do *not* sum to 1 (e.g. drop the normalization). Confirm the one-step residual is no longer ~1e-16, and identify which term in the formal definition broke.
- Exercise 2: Give clients *different* starting points instead of the shared $w_t$. Verify the one-step equivalence now fails too, isolating "shared initialization" as the load-bearing assumption.

## Further reading
- McMahan et al., "Communication-Efficient Learning of Deep Networks from Decentralized Data," AISTATS 2017 (arXiv:1602.05629), §2-3.
- `02-math-deep-dive.md`, §3 (one-step equivalence and the $\Delta=\eta^2\mathrm{Cov}_k(H_k,g_k)$ heterogeneity gap).


In [ ]:
"""Gradient- vs Model-Averaging Equivalence -- concept 05-gradient-model-averaging-equivalence of the paper learning map.

One local gradient step then size-weighted model averaging equals one averaged-gradient
step, because the weights n_k/n sum to 1 (Eq. 3.2); with TWO local steps the two schemes
diverge by a nonzero residual.
Runnable code analog of concepts/05-gradient-model-averaging-equivalence.md.
"""
import numpy as np


def main():
    np.random.seed(0)
    d = 5          # parameter dimension
    K = 4          # number of clients
    eta = 0.1      # learning rate

    # Shared start, client weights n_k/n (a probability vector summing to 1),
    # and a distinct quadratic local objective F_k per client: grad = A_k w - b_k.
    w_t = np.random.randn(d)
    n_k = np.array([100, 200, 50, 150], dtype=float)
    alpha = n_k / n_k.sum()                 # weights n_k / n; sum = 1
    A = [np.diag(0.5 + np.random.rand(d)) for _ in range(K)]   # SPD-ish local Hessians
    b = [np.random.randn(d) for _ in range(K)]

    def grad(k, w):
        return A[k] @ w - b[k]

    print("Concept 05: model-average of one local step == gradient-average step (Eq. 3.2)")
    print("Setup: d=%d params, K=%d clients, eta=%.2f, sum(alpha)=%.1f" % (d, K, eta, alpha.sum()))

    # --- ONE local step --------------------------------------------------------
    # Scheme A (average gradients, then step): w_t - eta * sum_k alpha_k g_k
    g_avg = sum(alpha[k] * grad(k, w_t) for k in range(K))
    wA_1 = w_t - eta * g_avg
    # Scheme B (step locally, then average models): sum_k alpha_k (w_t - eta g_k)
    wB_1 = sum(alpha[k] * (w_t - eta * grad(k, w_t)) for k in range(K))
    res1 = np.linalg.norm(wA_1 - wB_1)
    print("\nONE local step:")
    print("  ||model-average - gradient-average|| = %.3e  (should be ~1e-15)" % res1)
    print("  vectors equal at 1e-12 tol? %s" % np.allclose(wA_1, wB_1, atol=1e-12))

    # --- TWO local steps -------------------------------------------------------
    # Scheme A': two genuine centralized averaged-gradient steps on f = sum_k alpha_k F_k.
    wA = w_t.copy()
    for _ in range(2):
        gA = sum(alpha[k] * grad(k, wA) for k in range(K))
        wA = wA - eta * gA
    # Scheme B': each client runs TWO local steps from w_t, THEN average the models (Eq. 3.4).
    wB = np.zeros(d)
    for k in range(K):
        wk = w_t.copy()
        for _ in range(2):
            wk = wk - eta * grad(k, wk)
        wB += alpha[k] * wk
    res2 = np.linalg.norm(wA - wB)
    print("\nTWO local steps:")
    print("  ||model-average - gradient-average|| = %.3e  (nonzero: schemes diverge)" % res2)
    print("  vectors equal at 1e-12 tol? %s" % np.allclose(wA, wB, atol=1e-12))
    print("\nWitness: equivalence holds at one step (linearity + sum alpha=1);")
    print("the second local step is nonlinear in w_t, so averaging no longer commutes.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# FedAvg: Iterated Local Steps
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `06-fedavg-local-iteration`
**Graph:** `paper`
**Prerequisites:** [05-gradient-model-averaging-equivalence](05-gradient-model-averaging-equivalence.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg keeps the shared start $w_t$ but lets each client take $\tau\ge1$ local gradient steps before the server averages the endpoints. At $\tau=1$ this is exactly the one-step equivalence you saw in node 05 (averaging gradients = averaging models). For $\tau\ge2$ the per-client trajectories curve, so averaging their endpoints is no longer equal to running $\tau$ centralized GD steps — and that controlled mismatch is where the communication savings come from.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w_t$ | "w-sub-t" | Shared model broadcast at round $t$; the common start $w^k_{(0)}=w_t$ <!-- TODO add to foundations --> |
| $w^k_{(s)}$ | "w-sub-paren-s superscript k" | Client $k$'s local iterate after $s$ local steps <!-- TODO add to foundations --> |
| $\tau$ | "tau" | Number of local steps per client per round ($\ge1$) <!-- TODO add to foundations --> |
| $\eta$ | "eta" | Learning rate <!-- TODO add to foundations --> |
| $F_k(w)$ | "cap-F-sub-k of w" | Client $k$'s local objective <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ (weight $n_k/n$) <!-- TODO add to foundations --> |
| $H_k$ | "H-sub-k" | Local Hessian $\nabla^2 F_k(w_t)$ <!-- TODO add to foundations --> |
| $g_k$ | "g-sub-k" | Local gradient $\nabla F_k(w_t)$ <!-- TODO add to foundations --> |

## Formal definition
$$
w^{k}_{(s+1)}=w^{k}_{(s)}-\eta\,\nabla F_k\!\bigl(w^{k}_{(s)}\bigr),\quad w^k_{(0)}=w_t,\qquad
w_{t+1}=\sum_{k}\frac{n_k}{n}\,w^{k}_{(\tau)}.\tag{3.3}
$$
For $\tau\ge2$ the composite local map is nonlinear in $w_t$, so $\sum_k\frac{n_k}{n}G_k^{(\tau)}(w_t)\ne G^{(\tau)}(w_t)$ (centralized GD). The leading deviation is
$$
\Delta=\eta^2\Bigl(\textstyle\sum_k\frac{n_k}{n}H_k g_k-H\nabla f\Bigr)+O(\eta^3)=\eta^2\,\mathrm{Cov}_k(H_k,g_k)+O(\eta^3),\quad \Delta\equiv 0\text{ at }\tau=1.\tag{3.5}
$$

## Why this matters
This is the core FedAvg algorithm: Eq. (3.3) of `02-math-deep-dive.md` §3. The extra local steps ($\tau\ge2$) are exactly where communication savings come from, and the drift $\Delta$ of Eq. (3.5) is the object that `05-improvements.tex` M.1 cancels with control variates.

## Code
The aligned runnable demo lives at [`../code/06-fedavg-local-iteration.py`](../code/06-fedavg-local-iteration.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
   1  | 5.551115e-17
   2  | 1.107564e-03
   3  | 3.153887e-03
```

## Try it yourself
- Exercise 1: Set both clients' Hessians equal ($A_1=A_2$) and confirm the discrepancy collapses toward 0 for all $\tau$ — the homogeneous/IID limit where $\mathrm{Cov}_k(H_k,g_k)=0$.
- Exercise 2: Shrink $\eta$ by 10x and check the $\tau=2$ gap scales like $\eta^2$, matching Eq. (3.5).

## Further reading
- McMahan et al., *Communication-Efficient Learning of Deep Networks from Decentralized Data*, AISTATS 2017 (arXiv:1602.05629), §3.


In [ ]:
"""FedAvg: Iterated Local Steps -- concept 06-fedavg-local-iteration of the paper learning map.

FedAvg runs tau local gradient steps per client from a shared w_t, then averages
the endpoints (Eq. 3.3). For tau>=2 that round map is NOT tau centralized GD steps;
this witness measures the growing discrepancy (exactly 0 at tau=1) on a 2-client quadratic.
Runnable code analog of concepts/06-fedavg-local-iteration.py.
"""
import numpy as np


def make_quadratic(A, b):
    """Local objective F_k(w) = 0.5 w'Aw - b'w, so grad F_k(w) = A w - b."""
    return lambda w: A @ w - b


def local_run(grad, w0, eta, tau):
    """tau local GD steps from shared start w0 (one client's trajectory)."""
    w = w0.copy()
    for _ in range(tau):
        w = w - eta * grad(w)
    return w


def main():
    np.random.seed(0)
    d = 3
    eta = 0.05
    # Two heterogeneous clients: distinct local curvatures (A_k) and optima.
    A1 = np.array([[2.0, 0.3, 0.0], [0.3, 1.5, 0.1], [0.0, 0.1, 1.0]])
    A2 = np.array([[0.7, -0.2, 0.0], [-0.2, 2.5, 0.4], [0.0, 0.4, 1.8]])
    b1 = np.array([1.0, -0.5, 0.3])
    b2 = np.array([-0.4, 0.8, -1.1])
    n1, n2 = 3, 2                       # client sizes -> weights n_k/n
    n = n1 + n2
    a1, a2 = n1 / n, n2 / n

    g1, g2 = make_quadratic(A1, b1), make_quadratic(A2, b2)
    # Global objective f = sum_k (n_k/n) F_k  =>  grad f(w) = a1*g1(w) + a2*g2(w).
    grad_f = lambda w: a1 * g1(w) + a2 * g2(w)

    w_t = np.array([0.4, -0.2, 0.6])   # shared per-round start w^k_(0) = w_t

    print("FedAvg iterated local steps vs. centralized GD (2-client quadratic, eta=%.2f)" % eta)
    print("  FedAvg round: each client does tau local steps from shared w_t, then average endpoints.")
    print("  Centralized:  tau full GD steps on f = (n1/n)F_1 + (n2/n)F_2.")
    print("  Discrepancy = ||w_FedAvg - w_centralized||  (must be 0 at tau=1, grows with tau).")
    print("  tau | discrepancy ||FedAvg - centralized||")
    print("  ----+--------------------------------------")
    for tau in (1, 2, 3):
        # FedAvg: average the per-client endpoints (Eq. 3.3).
        w_fedavg = a1 * local_run(g1, w_t, eta, tau) + a2 * local_run(g2, w_t, eta, tau)
        # Centralized: tau genuine GD steps on the global f.
        w_central = local_run(grad_f, w_t, eta, tau)
        disc = np.linalg.norm(w_fedavg - w_central)
        print("   %d  | %.6e" % (tau, disc))

    print("Reading: tau=1 averaging gradients == averaging models (discrepancy 0);")
    print("for tau>=2 the nonlinear local trajectory diverges -- the gap IS the local-step free lunch.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Local-Update Count u = nE/(KB)

**Level:** `paper`
**Concept ID:** `07-local-update-count`
**Graph:** `paper`
**Prerequisites:** [06-fedavg-local-iteration](06-fedavg-local-iteration.md)
**Used by:** downstream nodes

## Plain-English intro
Each round, a client runs `ClientUpdate`: `E` passes (epochs) over its local data, each pass split into minibatches of size `B`, and every minibatch is one SGD step. So one client does `u_k = E*ceil(n_k/B)` local updates. Averaged over a uniformly random client (with `E[n_k]=n/K`), the expected per-round local work is `u = nE/(KB)`. This single number, not raw `(E,B)`, orders the configurations in the paper's Table 2 and quantifies how much extra on-device compute each round buys.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $u_k$ | "u-sub-k" | Local SGD updates client $k$ runs per round <!-- TODO add to foundations --> |
| $u$ | "u" | Expected local updates per round, over a random client <!-- TODO add to foundations --> |
| $E$ | "E" | Local epochs per client per round <!-- TODO add to foundations --> |
| $B$ | "B" | Local minibatch size ($B=\infty$ ⇒ full batch) <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $n$ | "n" | Total number of training examples <!-- TODO add to foundations --> |
| $K$ | "K" | Total number of clients <!-- TODO add to foundations --> |

## Formal definition
$$
u_k \;=\; E\Bigl\lceil\tfrac{n_k}{B}\Bigr\rceil
   \;\overset{B\,\mid\,n_k}{=}\; \frac{E\,n_k}{B}
\qquad\text{(Eq. 4.1)},
$$
$$
u \;:=\; \mathbb{E}[u_k] \;=\; \frac{E}{B}\,\mathbb{E}[n_k]
   \;=\; \frac{nE}{KB}\quad\bigl(\mathbb{E}[n_k]=\tfrac{n}{K}\bigr)
\qquad\text{(Eq. 4.2)}.
$$
The corner $B=\infty,\,E=1$ means one full-batch step, so $u_k=1$ (FedSGD) — not the naive $En_k/\infty=0$.

## Why this matters
$u=nE/(KB)$ is the statistic that orders the rows of Table 2 (e.g. MNIST CNN $n/K=600$: $E{=}1,B{=}50\Rightarrow u{=}12$; $E{=}20,B{=}10\Rightarrow u{=}1200$); it quantifies the compute-for-communication trade, appearing as Eqs. (4.1)-(4.2) of `02-math-deep-dive.md`.

## Code
The aligned runnable demo lives at [`../code/07-local-update-count.py`](../code/07-local-update-count.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Local-update count  u = nE/(KB)   (Eqs. 4.1-4.2 of the math deep dive)
K=100 clients, n=60000 total, balanced n_k=600 examples/client
FedSGD corner: B=inf, E=1  =>  u_k = 1 (one full-batch step)
```

## Try it yourself
- Exercise 1: Add the unbalanced case $B\nmid n_k$ (e.g. $n_k=600, B=64$) and watch the simulated count match $E\lceil n_k/B\rceil$, not the clean $En_k/B$.
- Exercise 2: Reproduce the full Table 2 ordering by sweeping $(E,B)$ and sorting configs by $u$; confirm $(20,10)$ is the heaviest at $u=1200$.

## Further reading
- McMahan et al., "Communication-Efficient Learning of Deep Networks from Decentralized Data" (arXiv:1602.05629), §3 / Table 2.
- `02-math-deep-dive.md` §4 (Eqs. 4.1-4.2).


In [ ]:
"""Local-Update Count u = nE/(KB) -- concept 07-local-update-count of the paper learning map.

Computes the per-round local SGD-step count u = nE/(KB) for several (E, B) and
confirms it against an actual ClientUpdate loop that counts the SGD steps taken.
Runnable code analog of concepts/07-local-update-count.md.
"""
import math

import numpy as np

np.random.seed(0)


def client_update_step_count(n_k, E, B):
    """Count SGD steps in ClientUpdate: E epochs, each over ceil(n_k/B) batches.

    B = math.inf means 'one batch = whole local set', i.e. full-batch (FedSGD).
    """
    if math.isinf(B):
        batches_per_epoch = 1  # full batch -> exactly one step per epoch
    else:
        batches_per_epoch = math.ceil(n_k / B)
    steps = 0
    for _epoch in range(E):
        # split the n_k local indices into batches; each batch = one SGD step
        idx = np.arange(n_k)
        np.random.shuffle(idx)
        for start in range(0, n_k, n_k if math.isinf(B) else B):
            steps += 1  # one minibatch -> one local SGD update
    assert steps == E * batches_per_epoch
    return steps


def main():
    K = 100          # number of clients
    n = 60000        # total examples (MNIST-scale)
    n_k = n // K     # balanced partition: 600 examples per client
    print("Local-update count  u = nE/(KB)   (Eqs. 4.1-4.2 of the math deep dive)")
    print("K=%d clients, n=%d total, balanced n_k=%d examples/client" % (K, n, n_k))
    print("FedSGD corner: B=inf, E=1  =>  u_k = 1 (one full-batch step)\n")

    configs = [(1, math.inf), (1, 50), (5, 10), (20, 10), (1, 600)]
    header = "  E |     B  | formula u=nE/(KB) | simulated steps/client | match"
    print(header)
    print("  " + "-" * (len(header) - 2))

    all_match = True
    for E, B in configs:
        # u = nE/(KB); for B=inf the paper's intended reading is u_k = E (one step/epoch)
        if math.isinf(B):
            u_formula = float(E)  # B=inf => one batch => E steps; E=1 => u=1 (FedSGD)
        else:
            u_formula = n * E / (K * B)
        simulated = client_update_step_count(n_k, E, B)
        ok = abs(simulated - u_formula) < 1e-9
        all_match = all_match and ok
        B_str = " inf" if math.isinf(B) else "%4d" % B
        print("  %2d | %s   |       %8.1f    |        %8d        |  %s"
              % (E, B_str, u_formula, simulated, "OK" if ok else "X"))

    print("\nFedSGD endpoint check: (E=1, B=inf) gives u_k = %d"
          % client_update_step_count(n_k, 1, math.inf))
    print("All formula values match the simulated ClientUpdate step counts: %s" % all_match)


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Heterogeneity Gap = $\eta^2\,\mathrm{Cov}_k(H_k, g_k)$
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `08-heterogeneity-gap-covariance`
**Graph:** `paper`
**Prerequisites:** [06-fedavg-local-iteration](06-fedavg-local-iteration.md), [Taylor theorem](https://github.com/pleyva2004/math-foundations/blob/main/concepts/taylor-theorem.md), [Hessian](https://github.com/pleyva2004/math-foundations/blob/main/concepts/hessian.md), [covariance](https://github.com/pleyva2004/math-foundations/blob/main/concepts/covariance.md)
**Used by:** downstream nodes

## Plain-English intro
Run two local gradient steps on each client, then average the models: this is *not* the same as two centralized gradient steps on the global objective. The difference is the **heterogeneity gap** $\Delta$. A Taylor expansion shows its leading term is $\eta^2$ times the client-weighted **covariance between local Hessians $H_k$ and local gradients $g_k$** — so the gap is zero only when clients are homogeneous, and grows with client drift otherwise.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w_t$ | "w-sub-t" | Shared per-round starting model <!-- TODO add to foundations --> |
| $w^k_{(2)}$ | "w-k-sub-2" | Client $k$'s model after 2 local steps <!-- TODO add to foundations --> |
| $G^{(2)}(w_t)$ | "cap-G-sup-2 of w-t" | Two centralized GD steps on $f$ <!-- TODO add to foundations --> |
| $g_k=\nabla F_k(w_t)$ | "g-sub-k" | Client $k$'s local gradient at $w_t$ <!-- TODO add to foundations --> |
| $H_k=\nabla^2 F_k(w_t)$ | "H-sub-k" | Client $k$'s local Hessian at $w_t$ <!-- TODO add to foundations --> |
| $H=\sum_k\frac{n_k}{n}H_k$ | "cap-H" | Weighted mean Hessian <!-- TODO add to foundations --> |
| $n_k/n$ | "n-k over n" | Client mixture weight <!-- TODO add to foundations --> |
| $\eta$ | "eta" | Learning rate <!-- TODO add to foundations --> |
| $\mathrm{Cov}_k(H_k,g_k)$ | "cov-sub-k" | $\sum_k\frac{n_k}{n}H_kg_k - H\,\nabla f$ <!-- TODO add to foundations --> |

## Formal definition
$$
\Delta \;=\; \sum_{k}\frac{n_k}{n}\,w^k_{(2)} \;-\; G^{(2)}(w_t)
\;=\; \eta^2\,\mathrm{Cov}_k\!\bigl(H_k,\,g_k\bigr) + O(\eta^3),
\qquad
\mathrm{Cov}_k(H_k,g_k)=\sum_{k}\tfrac{n_k}{n}H_k g_k - H\,\nabla f,\ \ H_k=\nabla^2 F_k.
$$

## Why this matters
This is Eq. (3.5) of `02-math-deep-dive.md`: the exact object behind FedAvg's client drift, and precisely what SCAFFOLD/FedProx (improvement M.1 in `05-improvements.tex`) are built to cancel.

## Code
The aligned runnable demo lives at [`../code/08-heterogeneity-gap-covariance.py`](../code/08-heterogeneity-gap-covariance.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Witness of Eq. (3.5): FedAvg(tau=2) - centralized(2 GD steps) = eta^2 Cov_k(H_k,g_k)
measured gap  Delta = avg_k w^k_(2) - G^(2)(w_t) = [-0.00150894 -0.000443   -0.00110393]
prediction eta^2 * Cov_k(H_k, g_k)              = [-0.00150894 -0.000443   -0.00110393]
```

## Try it yourself
- Exercise 1: Set all $H_k$ equal and all $b_k$ equal (homogeneous clients). Confirm $\Delta\to 0$ and $\mathrm{Cov}_k$ norm collapses.
- Exercise 2: Replace the quadratics with a cubic term so $H_k$ varies with $w$. Watch the residual $\|\Delta-\text{prediction}\|$ grow like $O(\eta^3)$ as you raise $\eta$.

## Further reading
- McMahan et al., *Communication-Efficient Learning of Deep Networks from Decentralized Data*, arXiv:1602.05629.
- Karimireddy et al., *SCAFFOLD* (control variates that cancel this $\mathrm{Cov}_k$ term).


In [ ]:
"""Heterogeneity Gap = eta^2 Cov_k(H_k, g_k) -- concept 08-heterogeneity-gap-covariance of the paper learning map.

Numerically witnesses Eq. (3.5): for tau=2 local steps the FedAvg-vs-centralized gap
equals eta^2 * weighted-Cov_k(H_k, g_k). With quadratic F_k the Hessian H_k is constant,
so the O(eta^3) remainder vanishes and the covariance prediction matches the measured gap.
Runnable code analog of concepts/08-heterogeneity-gap-covariance.py.
"""
import numpy as np


def main():
    np.random.seed(0)
    d, K = 3, 4                      # parameter dim, number of clients
    eta = 1e-2                       # learning rate
    n_k = np.array([10.0, 20.0, 30.0, 40.0])   # examples per client
    alpha = n_k / n_k.sum()          # mixture weights n_k / n

    # Per-client quadratics  F_k(w) = 1/2 w^T H_k w - b_k^T w  =>  grad g_k(w) = H_k w - b_k.
    # H_k constant (quadratic) so Taylor of (3.5) is exact through O(eta^2); O(eta^3) = 0.
    Hs, bs = [], []
    for _ in range(K):
        A = np.random.randn(d, d)
        Hs.append(A @ A.T + d * np.eye(d))     # SPD, heterogeneous per client
        bs.append(np.random.randn(d))
    Hs, bs = np.array(Hs), np.array(bs)

    w_t = np.random.randn(d)                    # shared per-round start

    def grad_k(k, w):
        return Hs[k] @ w - bs[k]

    # --- Local two-step endpoints w^k_(2), then weighted-average them (FedAvg, tau=2). ---
    w2 = np.zeros(d)
    for k in range(K):
        w = w_t - eta * grad_k(k, w_t)          # step 1
        w = w - eta * grad_k(k, w)              # step 2 (gradient at the moved point)
        w2 += alpha[k] * w
    # --- Two genuine *centralized* GD steps G^(2) on f = sum_k alpha_k F_k. ---
    H = np.tensordot(alpha, Hs, axes=1)         # H = sum_k alpha_k H_k
    b = alpha @ bs                              # so grad f(w) = H w - b
    g = w_t.copy()
    g = g - eta * (H @ g - b)
    g = g - eta * (H @ g - b)

    Delta = w2 - g                              # measured heterogeneity gap

    # --- Prediction: eta^2 * Cov_k(H_k, g_k) = eta^2 ( sum_k a_k H_k g_k - H grad_f ). ---
    g_k = np.array([grad_k(k, w_t) for k in range(K)])   # local grads at w_t
    mean_Hg = np.tensordot(alpha, np.einsum("kij,kj->ki", Hs, g_k), axes=1)
    grad_f = H @ w_t - b
    cov = mean_Hg - H @ grad_f                  # weighted covariance of (H_k, g_k)
    pred = eta ** 2 * cov

    print("Witness of Eq. (3.5): FedAvg(tau=2) - centralized(2 GD steps) = eta^2 Cov_k(H_k,g_k)")
    print("measured gap  Delta = avg_k w^k_(2) - G^(2)(w_t) =", np.round(Delta, 10))
    print("prediction eta^2 * Cov_k(H_k, g_k)              =", np.round(pred, 10))
    print("residual ||Delta - prediction|| =", np.linalg.norm(Delta - pred),
          "(== 0 to machine eps: quadratic => O(eta^3) term vanishes)")
    print("Cov_k norm =", round(float(np.linalg.norm(cov)), 6),
          "-> nonzero because clients are HETEROGENEOUS (this is client drift).")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Client-Sampling Unbiasedness (Horvitz-Thompson)
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `09-client-sampling-unbiasedness`
**Graph:** `paper`
**Prerequisites:** [03-iid-noniid-dichotomy](03-iid-noniid-dichotomy.md), [04-fedsgd-gradient-descent](04-fedsgd-gradient-descent.md), [unbiased estimator](https://github.com/pleyva2004/math-foundations/blob/main/concepts/unbiased-estimator.md)
**Used by:** downstream nodes

## Plain-English intro
When only a fraction $C<1$ of clients report each round, the server sees a random subset $S$, not everyone. If you weight each sampled client's gradient by the inverse of its chance of being picked, the random average lands *on the true full-data gradient in expectation* — no bias. This Horvitz-Thompson trick is what makes partial participation legitimate, and it needs no IID assumption: the unbiasedness is purely combinatorial.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $S,\ m=\lvert S\rvert$ | "S, m" | Random selected client set / its size, $m=\max(CK,1)$ <!-- TODO add to foundations --> |
| $K$ | "K" | Total number of clients <!-- TODO add to foundations --> |
| $p_k=m/K$ | "p-sub-k" | Inclusion probability of client $k$ (uniform sampling) <!-- TODO add to foundations --> |
| $n_k,\ n$ | "n-sub-k, n" | Examples on client $k$ / total examples <!-- TODO add to foundations --> |
| $g_k=\nabla F_k(w)$ | "g-sub-k" | Client $k$'s local average gradient at $w$ <!-- TODO add to foundations --> |
| $g(S)$ | "g of S" | Horvitz-Thompson sampled gradient estimator <!-- TODO add to foundations --> |
| $\nabla f(w)$ | "grad f" | True global gradient (the estimand) <!-- TODO add to foundations --> |

## Formal definition
Assume (A1): $S$ is a uniform $m$-subset of $[K]$ (sampling without replacement), so $p_k:=\Pr[k\in S]=m/K$ for every $k$. The Horvitz-Thompson / inverse-probability estimator weights each sampled term by $1/p_k$:
$$
g(S):=\sum_{k\in S}\frac{1}{p_k}\frac{n_k}{n}g_k=\frac{K}{m}\sum_{k\in S}\frac{n_k}{n}g_k,
\qquad
\mathbb{E}_S[g]=\sum_{k=1}^{K}p_k\cdot\frac{1}{p_k}\frac{n_k}{n}g_k=\sum_{k=1}^{K}\frac{n_k}{n}g_k=\nabla f(w).
$$
The $p_k$ cancels exactly (Eqs. 5.1-5.2). The uncorrected subset-sum $\sum_{k\in S}\frac{n_k}{n}g_k$ has expectation $\frac{m}{K}\nabla f(w)=C\,\nabla f(w)$ — biased toward $0$.

## Why this matters
Justifies partial participation ($C<1$): appears as Eqs. (5.1)-(5.2) of `02-math-deep-dive.md` §5, and underpins the unbiased-aggregation fix `05-improvements.tex` M.2(a). The unbiasedness is purely combinatorial — no IID needed; non-IID affects only $\mathrm{Var}(g)$, never $\mathbb{E}[g]$.

## Code
The aligned runnable demo lives at [`../code/09-client-sampling-unbiasedness.py`](../code/09-client-sampling-unbiasedness.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Client-Sampling Unbiasedness (Horvitz-Thompson), Eqs. 5.1-5.2
K=12 clients, m=4 sampled/round, inclusion prob p_k=m/K=0.333, MC trials=200000
   -> HT bias ||E[HT]-grad f|| = 1.0375e-03  (~0, UNBIASED)
```

## Try it yourself
- Exercise 1: Change $m$ (e.g. $m=2$ then $m=8$) and confirm the naive estimator's bias norm tracks the shrink factor $C=m/K$, while HT stays ~0.
- Exercise 2: Break (A1) by sampling clients with probability $\propto n_k$ instead of uniformly; show plain $\frac1m\sum_{k\in S}g_k$ then becomes unbiased while $K/m$-weighting does not (M.2(b)).

## Further reading
- McMahan et al. 2017, *Communication-Efficient Learning of Deep Networks from Decentralized Data*, footnote 2 (arXiv:1602.05629).
- Horvitz & Thompson 1952, *A generalization of sampling without replacement from a finite universe*, JASA.


In [ ]:
"""Client-Sampling Unbiasedness (Horvitz-Thompson) -- concept 09-client-sampling-unbiasedness of the paper learning map.

Monte-Carlo witness that the Horvitz-Thompson estimator g(S)=(K/m) sum_{k in S}(n_k/n)g_k
is unbiased for the true weighted sum sum_k (n_k/n) g_k = grad f(w), while a naive
uncorrected subset-sum is biased (it under-counts by the participation fraction m/K).
Runnable code analog of concepts/09-client-sampling-unbiasedness.py.
"""
import numpy as np


def main():
    rng = np.random.default_rng(0)
    np.random.seed(0)

    K = 12          # total clients
    m = 4           # clients sampled per round (C = m/K = 1/3)
    n_per = 50      # examples per client (balanced n_k = n/K)
    n_k = np.full(K, n_per, dtype=float)
    n = n_k.sum()
    p_k = m / K     # uniform inclusion probability (Eq. A1)

    # Each client's local average gradient g_k (deterministic given w); use d=3 vectors.
    g = rng.standard_normal((K, 3))

    # TRUE target: grad f(w) = sum_k (n_k/n) g_k  (the partition identity, Eq. 2.2).
    weights = n_k / n
    g_true = (weights[:, None] * g).sum(axis=0)

    trials = 200_000
    ht_acc = np.zeros(3)      # Horvitz-Thompson estimator accumulator (Eq. 5.2)
    naive_acc = np.zeros(3)   # naive uncorrected subset-sum: sum_{k in S}(n_k/n)g_k
    for _ in range(trials):
        S = np.random.choice(K, size=m, replace=False)  # uniform m-subset
        contrib = (n_k[S] / n)[:, None] * g[S]           # the (n_k/n) g_k terms on S
        ht_acc += (K / m) * contrib.sum(axis=0)          # 1/p_k = K/m correction
        naive_acc += contrib.sum(axis=0)                 # no 1/p_k correction -> biased

    ht_mean = ht_acc / trials
    naive_mean = naive_acc / trials

    ht_bias = np.linalg.norm(ht_mean - g_true)
    naive_bias = np.linalg.norm(naive_mean - g_true)
    # The naive estimator is the HT mean shrunk by p_k = m/K, so its target is (m/K)*g_true.
    predicted_naive = (m / K) * g_true

    print("Client-Sampling Unbiasedness (Horvitz-Thompson), Eqs. 5.1-5.2")
    print("K={} clients, m={} sampled/round, inclusion prob p_k=m/K={:.3f}, MC trials={}"
          .format(K, m, p_k, trials))
    print("true target  grad f(w) = sum_k (n_k/n) g_k        =", np.round(g_true, 4))
    print("E[HT estimator] (K/m)*sum_S (n_k/n)g_k            =", np.round(ht_mean, 4))
    print("   -> HT bias ||E[HT]-grad f|| = {:.4e}  (~0, UNBIASED)".format(ht_bias))
    print("E[naive subset-sum] sum_S (n_k/n)g_k              =", np.round(naive_mean, 4))
    print("   -> naive bias ||E[naive]-grad f|| = {:.4e}  (BIASED)".format(naive_bias))
    print("naive matches shrunk target (m/K)*grad f          =", np.round(predicted_naive, 4))
    print("   -> ||E[naive]-(m/K)grad f|| = {:.4e}  (confirms bias factor = C = m/K = {:.3f})"
          .format(np.linalg.norm(naive_mean - predicted_naive), m / K))

    assert ht_bias < 1e-2, "HT estimator should be unbiased"
    assert naive_bias > 0.1, "naive estimator should be visibly biased"
    print("VERDICT: PASS -- the 1/p_k=K/m correction makes the sampled gradient unbiased.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Aggregation Erratum: normalize by m_t
**Level:** `paper`
**Concept ID:** `10-aggregation-erratum`
**Graph:** `paper`
**Prerequisites:** [09-client-sampling-unbiasedness](09-client-sampling-unbiasedness.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg's server combines the selected clients' models into a weighted average. The published erratum (footnote 4) fixes the normalizer: divide each client's weight $n_k$ by $m_t$ (the example count *over the selected set*), not by the global $n$. Using $n$ makes the weights sum to only $m_t/n<1$, so the aggregate is shrunk toward the origin every round.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w^{k}$ | "w-super-k" | Client $k$'s locally updated model <!-- TODO add to foundations --> |
| $w_{t+1}$ | "w-sub-t-plus-one" | Server's aggregated model after round $t$ <!-- TODO add to foundations --> |
| $S_t$ | "S-sub-t" | Set of clients selected in round $t$ <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $n$ | "n" | Total number of training examples <!-- TODO add to foundations --> |
| $m_t=\sum_{k\in S_t}n_k$ | "m-sub-t" | Total examples on the selected clients <!-- TODO add to foundations --> |
| $C$ | "C" | Fraction of clients selected per round ($m/K$) <!-- TODO add to foundations --> |

## Formal definition
$$
\textbf{Correct (Eq. 6.1):}\quad w_{t+1}=\sum_{k\in S_t}\frac{n_k}{m_t}\,w^{k},\quad m_t=\sum_{k\in S_t}n_k,\qquad \sum_{k\in S_t}\frac{n_k}{m_t}=\frac{m_t}{m_t}=1.
$$
$$
\textbf{Buggy (Eq. 6.4):}\quad w_{t+1}^{\text{wrong}}=\sum_{k\in S_t}\frac{n_k}{n}\,w^{k}=\frac{m_t}{n}\,w_{t+1},\quad \sum_{k\in S_t}\frac{n_k}{n}=\frac{m_t}{n}<1,\qquad \mathbb{E}_{S_t}\!\big[w_{t+1}^{\text{wrong}}\big]=\frac{m}{K}\,\bar v=C\,\bar v\ \ (\text{Eq. 6.5}).
$$

## Why this matters
This is the published erratum (footnote 4), formalized as Eqs. (6.1)-(6.5) of `02-math-deep-dive.md` §6: the buggy $n_k/n$ rule multiplies the aggregate by $m_t/n\approx C$ every round, so the model decays geometrically like $C^t$.

## Code
The aligned runnable demo lives at [`../code/10-aggregation-erratum.py`](../code/10-aggregation-erratum.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
FedAvg aggregation erratum: normalize by m_t, not n  (deep dive Eqs. 6.1-6.5)
K=100  C=0.10  m=|S_t|=10  n_per=50  n=5000  m_t=sum n_k over S=500

CORRECT  weights n_k/m_t  sum = 1.000000   (= 1, a true convex combination)
```

## Try it yourself
- Exercise 1: Make the clients unbalanced (vary `n_per` per client). Confirm the correct weights still sum to 1, while the shrink factor $m_t/n$ now fluctuates around $C$ across re-samples of $S_t$.
- Exercise 2: Iterate the buggy rule for 20 rounds (feed `w_buggy` back as the next start) and plot $\|w_t\|$; verify the $\sim C^t$ geometric collapse.

## Further reading
- McMahan et al., *Communication-Efficient Learning of Deep Networks from Decentralized Data*, AISTATS 2017 (arXiv:1602.05629), Algorithm 1 and footnote 4.
- `02-math-deep-dive.md` §6 (Eqs. 6.1-6.5).


In [ ]:
"""Aggregation Erratum: normalize by m_t -- concept 10-aggregation-erratum of the paper learning map.

Shows that FedAvg's corrected server update normalizes by m_t = sum_{k in S} n_k (weights
sum to 1), whereas the buggy n_k/n rule shrinks each round's aggregate by m_t/n ~= C.
Runnable code analog of concepts/10-aggregation-erratum.md.
"""

import numpy as np


def main():
    np.random.seed(0)

    # ---- Setup: K clients, balanced for clean C = m/K, dim-d toy "models" ----
    K = 100          # total clients
    C = 0.1          # fraction selected per round
    m = max(int(C * K), 1)   # |S_t|
    d = 4            # model dimension
    n_per = 50       # examples per client (balanced)
    n = K * n_per    # total examples

    n_k_all = np.full(K, n_per)        # client sizes
    # one toy local model w^k per client (random but fixed by the seed)
    w_clients = np.random.randn(K, d)

    # ---- Round t: server samples a uniform m-subset S_t (without replacement) ----
    S = np.sort(np.random.choice(K, size=m, replace=False))
    n_k = n_k_all[S]                   # sizes of selected clients
    m_t = n_k.sum()                    # = sum_{k in S} n_k          (Eq. 6.1)
    w_sel = w_clients[S]               # selected local models

    # ---- Correct rule (Eq. 6.1): normalize by m_t -> weights sum to 1 ----
    weights_correct = n_k / m_t
    w_correct = (weights_correct[:, None] * w_sel).sum(axis=0)

    # ---- Buggy rule (Eq. 6.4): normalize by n -> weights sum to m_t/n < 1 ----
    weights_buggy = n_k / n
    w_buggy = (weights_buggy[:, None] * w_sel).sum(axis=0)

    shrink = m_t / float(n)            # the m_t/n shrink factor (Eq. 6.4)

    print("FedAvg aggregation erratum: normalize by m_t, not n  (deep dive Eqs. 6.1-6.5)")
    print("K=%d  C=%.2f  m=|S_t|=%d  n_per=%d  n=%d  m_t=sum n_k over S=%d"
          % (K, C, m, n_per, n, m_t))
    print()
    print("CORRECT  weights n_k/m_t  sum = %.6f   (= 1, a true convex combination)"
          % weights_correct.sum())
    print("BUGGY    weights n_k/n    sum = %.6f   (= m_t/n < 1, shrinks the model)"
          % weights_buggy.sum())
    print()
    print("shrink factor m_t/n = %.6f    (C = m/K = %.6f; equal in expectation, Eq. 6.5)"
          % (shrink, m / float(K)))
    print()
    # Witness: buggy aggregate == shrink * correct aggregate, component-wise (Eq. 6.4)
    recon = shrink * w_correct
    print("correct aggregate w_{t+1}      =", np.array2string(w_correct, precision=4))
    print("buggy aggregate (n_k/n)        =", np.array2string(w_buggy, precision=4))
    print("shrink * correct  (predicted)  =", np.array2string(recon, precision=4))
    print("max|buggy - shrink*correct|    = %.2e  -> buggy = (m_t/n) * correct, exactly"
          % np.max(np.abs(w_buggy - recon)))
    print()
    # Geometric decay: feeding w_buggy back as next start shrinks by ~C each round.
    norm0 = np.linalg.norm(w_correct)
    norms = [norm0 * shrink ** t for t in range(5)]
    print("If the buggy rule recurs, ||w_t|| decays like (m_t/n)^t ~ C^t (geometric collapse):")
    print("  ||w_t|| over rounds 0..4 = " +
          ", ".join("%.4f" % v for v in norms))


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Parameter Averaging & Jensen
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `11-parameter-averaging-jensen`
**Graph:** `paper`
**Prerequisites:** [02-client-partition-decomposition](02-client-partition-decomposition.md), [convex function](https://github.com/pleyva2004/math-foundations/blob/main/concepts/convex-function.md), [Jensen's inequality](https://github.com/pleyva2004/math-foundations/blob/main/concepts/jensens-inequality.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg's server step replaces every client's model with one weighted average $\bar w=\sum_k\beta_k w^k$. If the loss $f$ is convex, Jensen's inequality guarantees this averaged model is never worse than the weighted-mean of the parents, and in turn no worse than the *worst* parent. That is the clean reason averaging in parameter space is "safe" in the convex case -- and the reason the non-convex case (real nets) instead needs the shared-initialization trick.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $f(w)$ | "f of w" | Global (convex) objective being averaged over <!-- TODO add to foundations --> |
| $w^k$ | "w-super-k" | Client $k$'s locally-trained model <!-- TODO add to foundations --> |
| $\beta_k=n_k/m_t$ | "beta-sub-k" | Mixture weight; probability vector ($\beta_k\ge0,\sum_k\beta_k=1$) <!-- TODO add to foundations --> |
| $F_k$ | "cap-F-sub-k" | Client $k$'s local objective; $f=\sum_k\alpha_k F_k$ <!-- TODO add to foundations --> |
| $\bar w=\sum_k\beta_k w^k$ | "w-bar" | The averaged (aggregated) model <!-- TODO add to foundations --> |

## Formal definition
For convex $F_k$, the mixture $f=\sum_k\alpha_k F_k$ is convex, so for any convex weights $\beta_k\ge0,\ \sum_k\beta_k=1$:
$$ f\!\Big(\sum_{k}\beta_k w^k\Big)\ \le\ \sum_{k}\beta_k\,f(w^k)\ \le\ \max_k f(w^k). $$
The first inequality is Jensen; the second is that a convex combination is bounded by its largest term.

## Why this matters
Appears as Eq. (7.1) of `02-math-deep-dive.md` (§7a). It formalizes why averaging is harmless under convexity and motivates §7b-c -- non-convex nets have loss barriers, so FedAvg must engineer a shared per-round init to keep the parents in a common basin.

## Code
The aligned runnable demo lives at [`../code/11-parameter-averaging-jensen.py`](../code/11-parameter-averaging-jensen.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.
**Expected output preview:**
```
Jensen chain on a convex quadratic f (Eq. 7.1), K=5 clients, d=4:
  beta (mixture weights, sum=1.000): [0.34  0.113 0.078 0.339 0.13 ]
  per-client losses f(w^k):        [87.6346 72.7985 83.4229 67.1886 44.7001]
```

## Try it yourself
- Exercise 1: Make $A$ negative-definite (non-convex $f$) and watch the Jensen assertion fail -- the averaged model can exceed every parent.
- Exercise 2: Add the strong-convexity midpoint refinement $f(\tfrac{w+w'}2)\le\tfrac12 f(w)+\tfrac12 f(w')-\tfrac\mu8\lVert w-w'\rVert^2$ and verify the $\mu$-gain numerically.

## Further reading
- `02-math-deep-dive.md` §7 (convex vs non-convex averaging); Jensen's inequality (any convex-analysis text).


In [ ]:
"""Parameter Averaging & Jensen -- concept 11-parameter-averaging-jensen of the paper learning map.

On a convex quadratic f, sample several client models w^k and verify numerically the
Jensen chain f(sum_k beta_k w^k) <= sum_k beta_k f(w^k) <= max_k f(w^k) (Eq. 7.1):
why averaging is safe in the convex case (and thus why non-convex needs shared init).
Runnable code analog of concepts/11-parameter-averaging-jensen.md.
"""
import numpy as np


def main():
    np.random.seed(0)

    # A convex quadratic global objective f(w) = 0.5 (w-c)^T A (w-c) + const.
    # A is SPD => f is convex (Hessian = A >= 0). d = parameter dimension.
    d = 4
    M = np.random.randn(d, d)
    A = M @ M.T + d * np.eye(d)          # symmetric positive definite
    c = np.random.randn(d)               # minimizer location

    def f(w):
        r = w - c
        return 0.5 * float(r @ A @ r)

    # K client models w^k, scattered around the optimum (the FedAvg parents).
    K = 5
    W = c + 1.5 * np.random.randn(K, d)  # rows are the w^k

    # Mixture weights beta_k = n_k / m_t: a probability vector (beta_k >= 0, sum = 1).
    n_k = np.random.randint(50, 500, size=K).astype(float)
    beta = n_k / n_k.sum()

    # LHS: loss of the weighted-averaged model (what the server broadcasts).
    w_bar = beta @ W                      # sum_k beta_k w^k
    lhs = f(w_bar)

    # MIDDLE: weighted average of the per-client losses.
    fk = np.array([f(W[k]) for k in range(K)])
    mid = float(beta @ fk)

    # RHS: the worst client's loss.
    rhs = float(fk.max())

    print("Jensen chain on a convex quadratic f (Eq. 7.1), K=%d clients, d=%d:" % (K, d))
    print("  beta (mixture weights, sum=%.3f): %s" % (beta.sum(), np.round(beta, 3)))
    print("  per-client losses f(w^k):        %s" % np.round(fk, 4))
    print("  LHS  f(avg model)            = %.6f" % lhs)
    print("  MID  sum_k beta_k f(w^k)     = %.6f" % mid)
    print("  RHS  max_k f(w^k)            = %.6f" % rhs)

    eps = 1e-9
    jensen_ok = lhs <= mid + eps
    max_ok = mid <= rhs + eps
    print("  Jensen   LHS <= MID ? %s   (gap MID-LHS = %.6f >= 0)" % (jensen_ok, mid - lhs))
    print("  Bound    MID <= RHS ? %s   (slack RHS-MID = %.6f >= 0)" % (max_ok, rhs - mid))

    assert jensen_ok, "Jensen inequality violated -- f should be convex"
    assert max_ok, "weighted mean exceeds the max -- impossible for a convex combination"
    print("VERDICT: PASS -- averaging the convex parents is no worse than the worst parent.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Shared-Init & the Permutation Barrier
**Level:** `paper`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `12-shared-init-permutation-barrier`
**Graph:** `paper`
**Prerequisites:** [11-parameter-averaging-jensen](11-parameter-averaging-jensen.md), [permutation group](https://github.com/pleyva2004/math-foundations/blob/main/concepts/permutation-group.md)
**Used by:** downstream nodes

## Plain-English intro
A one-hidden-layer network computes the *same function* if you relabel its hidden units, so each minimizer is copied across a huge orbit of permuted twins. Two nets trained from *independent* random seeds land in *incompatible* twins, and naively averaging their weights blends unrelated units, producing a model worse than both parents (a loss "barrier"). FedAvg dodges this by broadcasting the *same* model $w_t$ every round, anchoring all clients in one basin so the average instead beats both.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $W_1, b_1$ | "W-one, b-one" | First-layer weights / bias (hidden layer) <!-- TODO add to foundations --> |
| $W_2$ | "W-two" | Second-layer (output) weights <!-- TODO add to foundations --> |
| $P$ | "P" | A hidden-unit permutation matrix <!-- TODO add to foundations --> |
| $\sigma$ | "sigma" | Elementwise nonlinearity (e.g. $\tanh$) <!-- TODO add to foundations --> |
| $w_t$ | "w-sub-t" | Shared global model broadcast at round $t$ <!-- TODO add to foundations --> |
| $w, w'$ | "w, w-prime" | Two client models being averaged <!-- TODO add to foundations --> |

## Formal definition
A 1-hidden-layer net is invariant under any hidden-unit permutation $P$ (an element of the symmetric group $S_h$ acting as a permutation matrix):
$$ W_2\,\sigma(W_1 x + b_1)=(W_2 P^\top)\,\sigma\bigl(P W_1 x + P b_1\bigr),\qquad (W_1,b_1,W_2)\mapsto(W_1 P^\top,\,P b_1,\,P W_2). $$
Hence every minimizer is replicated across an orbit of size $\prod_\ell |S_{h_\ell}|$. Independent inits occupy incompatible orbit elements, so $f\bigl(\tfrac12 w+\tfrac12 w'\bigr)>\max\{f(w),f(w')\}$ (Fig. 1 barrier); a shared $w_t$ fixes one orbit element, keeping clients in a common basin where averaging instead lowers loss.

## Why this matters
This is the load-bearing empirical justification for FedAvg, formalized in §7(b)-(c) of `02-math-deep-dive.md`; it motivates improvement T.1 (permutation-aligned averaging) in `05-improvements.tex`.

## Code
The aligned runnable demo lives at [`../code/12-shared-init-permutation-barrier.py`](../code/12-shared-init-permutation-barrier.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.
**Expected output preview:**
```
FedAvg Fig.1: averaging two 1-hidden-layer MLPs trained on disjoint halves.
Barrier height = loss(average) - max(loss(parents)); >0 means avg is WORSE.

INDEPENDENT inits : parents=(0.1982, 0.1849)  avg=0.2231  barrier=+0.0249  -> BARRIER (avg worse)
SHARED init  : parents=(0.1958, 0.1798)  avg=0.1778  barrier=-0.0180  -> VALLEY (avg better)
```

## Try it yourself
- Exercise 1: Permute one trained model's hidden units by a random $P$ (apply $W_1\mapsto W_1 P^\top, b_1\mapsto P b_1, W_2\mapsto P W_2$); confirm `loss` is unchanged to ~1e-9, then average it with the original and watch the barrier reappear.
- Exercise 2: Sweep the number of hidden units $H$. Does a larger (more over-parameterized) net make the shared-init valley deeper and the independent-init barrier higher?

## Further reading
- McMahan et al., *Communication-Efficient Learning of Deep Networks from Decentralized Data*, AISTATS 2017 (Fig. 1).
- Ainsworth, Hayase, Srinivasa, *Git Re-Basin: Merging Models modulo Permutation Symmetries*, ICLR 2023.


In [ ]:
"""Shared-Init & the Permutation Barrier -- concept 12-shared-init-permutation-barrier of the paper learning map.

A 1-hidden-layer MLP's function is invariant under permuting its hidden units, so
independent inits land in incompatible orbit elements and naive averaging crosses a
loss barrier (avg worse than both parents); a SHARED init keeps both in one basin so
the average beats both. Runnable code analog of concepts/12-shared-init-permutation-barrier.md.
"""
import numpy as np

np.random.seed(0)
np.seterr(all="ignore")  # silence spurious BLAS matmul FPE warnings (numpy 2.0.x; values stay finite)


def make_data(n=400, d=6):
    X = np.random.randn(n, d)
    y = (X @ np.random.randn(d) + 0.3 * np.random.randn(n) > 0).astype(np.float64)
    return X, y


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


def forward(p, X):                                   # invariant under hidden-unit perm
    return sigmoid(np.tanh(X @ p["W1"] + p["b1"]) @ p["W2"] + p["b2"]).ravel()


def loss(p, X, y):
    yh = np.clip(forward(p, X), 1e-9, 1 - 1e-9)
    return float(-np.mean(y * np.log(yh) + (1 - y) * np.log(1 - yh)))


def init(d, H):
    return {"W1": np.random.randn(d, H) * 0.5, "b1": np.zeros(H),
            "W2": np.random.randn(H) * 0.5, "b2": 0.0}


def train(p, X, y, epochs=400, eta=0.5):
    p = {k: (v.copy() if hasattr(v, "copy") else v) for k, v in p.items()}
    n = len(y)
    for _ in range(epochs):
        h = np.tanh(X @ p["W1"] + p["b1"])
        e = (sigmoid(h @ p["W2"] + p["b2"]).ravel() - y) / n
        dh = np.outer(e, p["W2"]) * (1 - h ** 2)
        p["W2"] -= eta * (h.T @ e)
        p["b2"] -= eta * float(e.sum())
        p["W1"] -= eta * (X.T @ dh)
        p["b1"] -= eta * dh.sum(axis=0)
    return p


def avg(pa, pb):
    return {k: 0.5 * (pa[k] + pb[k]) for k in pa}


def barrier(shared, X, y, d, H):
    p0a = init(d, H)
    p0b = p0a if shared else init(d, H)              # SAME or independent seed
    half = len(y) // 2
    pa = train(p0a, X[:half], y[:half])              # disjoint halves
    pb = train(p0b, X[half:], y[half:])
    la, lb = loss(pa, X, y), loss(pb, X, y)          # full-set loss of each parent
    lavg = loss(avg(pa, pb), X, y)                    # full-set loss of average
    return lavg - max(la, lb), la, lb, lavg


def main():
    X, y = make_data()
    d, H = X.shape[1], 8
    print("FedAvg Fig.1: averaging two 1-hidden-layer MLPs trained on disjoint halves.")
    print("Barrier height = loss(average) - max(loss(parents)); >0 means avg is WORSE.\n")
    for shared in (False, True):
        bh, la, lb, lavg = barrier(shared, X, y, d, H)
        tag = "SHARED init " if shared else "INDEPENDENT inits"
        verdict = "BARRIER (avg worse)" if bh > 0 else "VALLEY (avg better)"
        print("%s : parents=(%.4f, %.4f)  avg=%.4f  barrier=%+.4f  -> %s"
              % (tag, la, lb, lavg, bh, verdict))
    print("\nWitness: independent inits occupy incompatible permutation-orbit elements")
    print("(barrier > 0); a shared init keeps both clients in one basin (barrier < 0).")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Control-Variate Drift Correction
**Level:** `extension`
**Concept ID:** `01-control-variate-drift-correction`
**Graph:** `improvements`
**Prerequisites:** [paper:08-heterogeneity-gap-covariance](../../paper/concepts/08-heterogeneity-gap-covariance.md), [control variates](https://github.com/pleyva2004/math-foundations/blob/main/concepts/control-variates.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg lets each client take several local gradient steps before averaging, but on non-IID data each client drifts toward *its own* optimum instead of the global one. A control variate is a per-client correction vector that, added to every local step, re-centers that client's trajectory onto the global gradient direction. This is the SCAFFOLD fix, and it directly cancels the heterogeneity drift identified in paper node 08.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w$ | "w" | Model parameter vector being updated locally <!-- TODO add to foundations --> |
| $\eta$ | "eta" | Local learning rate <!-- TODO add to foundations --> |
| $g_{\mathrm{local}}(w)$ | "g-local of w" | Client's local gradient at the current local iterate $w$ <!-- TODO add to foundations --> |
| $c_k$ | "c-sub-k" | Client $k$'s control variate (its drift estimate) <!-- TODO add to foundations --> |
| $c$ | "c" | Server control variate, average of selected $c_k$ <!-- TODO add to foundations --> |
| $S_t$ | "S-sub-t" | Set of clients selected in round $t$ <!-- TODO add to foundations --> |
| $H_k=\nabla^2 F_k(w_t)$ | "H-sub-k" | Client $k$'s local Hessian at $w_t$ <!-- TODO add to foundations --> |
| $g_k=\nabla F_k(w_t)$ | "g-sub-k" | Client $k$'s local gradient at the shared point <!-- TODO add to foundations --> |

## Formal definition
$$
\text{Local step (client } k):\quad w \leftarrow w-\eta\bigl(g_{\mathrm{local}}(w)-c_k+c\bigr),
\qquad c=\frac{1}{|S_t|}\sum_{k\in S_t}c_k .
$$
In expectation the $-c_k+c$ term replaces the locally-biased direction with the global one, cancelling the leading client-heterogeneity drift
$$
\Delta=\eta^2\,\mathrm{Cov}_k\!\bigl(H_k,g_k\bigr)+O(\eta^3),
$$
so the multi-step round recovers $w\leftarrow w-\eta\,\nabla f(w)$ to leading order.

## Why this matters
Operationalizes the heterogeneity-gap finding (paper node 08): it cancels the leading $\eta^2\mathrm{Cov}_k(H_k,g_k)$ drift of Eq. (3.5) in `02-math-deep-dive.md`, and is the highest-leverage proposal M.1 of `05-improvements.tex`.

## Code
The aligned runnable demo lives at [`../code/01-control-variate-drift-correction.py`](../code/01-control-variate-drift-correction.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Global gradient grad f(w_t) at shared point w_t=[0.7 0.7]:
  g_global = [-0.275 -0.275]
Per-client effective local direction vs global gradient:
```

## Try it yourself
- Exercise 1: Set the two Hessians equal (`H[1] = H[0]`) — the homogeneous limit. Confirm the plain and corrected deviations both collapse, since $\mathrm{Cov}_k(H_k,g_k)=0$.
- Exercise 2: Increase `tau` (local steps) and `eta`. Watch the *plain* drift grow with $\eta^2\tau$ while the corrected direction stays anchored to the global gradient.

## Further reading
- Karimireddy et al., "SCAFFOLD: Stochastic Controlled Averaging for Federated Learning," ICML 2020.


In [ ]:
"""Control-Variate Drift Correction -- concept 01-control-variate-drift-correction
of the improvements learning map.

On a 2-client heterogeneous quadratic, SCAFFOLD-style control variates re-center
each client's multi-step local trajectory onto the GLOBAL gradient direction,
cancelling the leading client-heterogeneity drift eta^2*Cov_k(H_k, g_k) (eq. 3.5
of 02-math-deep-dive.md; M.1 of 05-improvements.tex).

Runnable code analog of concepts/01-control-variate-drift-correction.py
(see concepts/01-control-variate-drift-correction.md).
"""
import numpy as np


def main():
    np.random.seed(0)

    # Two heterogeneous local quadratics  F_k(w) = 0.5 (w-b_k)^T H_k (w-b_k).
    # grad F_k(w) = H_k (w - b_k);  Hessians H_k differ => heterogeneity.
    H = [np.array([[3.0, 0.0], [0.0, 0.5]]),
         np.array([[0.5, 0.0], [0.0, 3.0]])]
    b = [np.array([1.0, 0.0]), np.array([0.0, 1.0])]
    nk = np.array([1.0, 1.0])          # equal client sizes -> weights n_k/n = 1/2
    alpha = nk / nk.sum()

    def gk(k, w):                      # local gradient g_local(w) for client k
        return H[k] @ (w - b[k])

    w0 = np.array([0.7, 0.7])          # shared broadcast point w_t
    eta, tau = 0.10, 5                 # local lr and number of local steps

    # GLOBAL gradient at the shared point: grad f(w0) = sum_k alpha_k g_k(w0)  (eq. 2.2)
    g_global = sum(alpha[k] * gk(k, w0) for k in range(2))

    # Control variates: c_k = g_k(w0), server c = mean_k c_k  (eq. M.1).
    c = [gk(k, w0) for k in range(2)]
    c_server = sum(alpha[k] * c[k] for k in range(2))

    def local_traj(k, corrected):
        w = w0.copy()
        for _ in range(tau):
            d = gk(k, w)
            if corrected:
                d = d - c[k] + c_server     # heterogeneity-corrected direction
            w = w - eta * d
        return (w0 - w) / (tau * eta)        # avg per-step direction client moved

    # Compare each client's effective local direction to the global gradient.
    print("Global gradient grad f(w_t) at shared point w_t={}:".format(w0))
    print("  g_global = {}".format(np.round(g_global, 4)))
    print("Per-client effective local direction vs global gradient:")
    dev_plain, dev_corr = 0.0, 0.0
    for k in range(2):
        dp = local_traj(k, corrected=False)
        dc = local_traj(k, corrected=True)
        ep = np.linalg.norm(dp - g_global)
        ec = np.linalg.norm(dc - g_global)
        dev_plain += alpha[k] * ep
        dev_corr += alpha[k] * ec
        print("  client {}: plain dir ={}  dev={:.4f} | corrected dir ={}  dev={:.4f}"
              .format(k, np.round(dp, 3), ep, np.round(dc, 3), ec))

    print("Client-weighted deviation from global gradient:")
    print("  plain local steps      : {:.4f}".format(dev_plain))
    print("  control-variate steps  : {:.4f}".format(dev_corr))
    assert dev_corr < dev_plain, "corrected direction must track global gradient better"
    print("VERDICT: PASS -- control variates make local steps track grad f "
          "({:.2f}x smaller drift).".format(dev_plain / dev_corr))


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Unbiased Aggregation Estimators
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `02-unbiased-aggregation-estimators`
**Graph:** `improvements`
**Prerequisites:** [paper:10-aggregation-erratum](../../paper/concepts/10-aggregation-erratum.md), [Horvitz-Thompson estimator](https://github.com/pleyva2004/math-foundations/blob/main/concepts/horvitz-thompson-estimator.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg's corrected server step averages the selected clients' models weighted by their data counts and normalized by the total examples *in the drawn set*. Because that normalizer is itself random and tends to be large exactly when a big client is sampled, the average systematically over-weights large clients — it is a Hajek *ratio* estimator, biased under client imbalance. Swapping the random normalizer for the fixed inclusion probability (Horvitz-Thompson) restores exact unbiasedness for any imbalance.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $v_k$ | "v-sub-k" | Client $k$'s returned model $w^k_{t+1}$ <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $n=\sum_k n_k$ | "n" | Total examples over all $K$ clients <!-- TODO add to foundations --> |
| $S$ | "S" | Random set of selected clients, $\lvert S\rvert=m$ <!-- TODO add to foundations --> |
| $m_t=\sum_{k\in S} n_k$ | "m-sub-t" | Random normalizer: examples on the selected clients <!-- TODO add to foundations --> |
| $m,\,K$ | "m, K" | Clients sampled per round; total clients <!-- TODO add to foundations --> |
| $p_k=m/K$ | "p-sub-k" | Uniform inclusion probability of client $k$ <!-- TODO add to foundations --> |
| $\bar v$ | "v-bar" | Target: full size-weighted mean $\sum_k\frac{n_k}{n}v_k$ <!-- TODO add to foundations --> |

## Formal definition
$$
\bar v=\sum_{k=1}^{K}\tfrac{n_k}{n}\,v_k,\qquad
\underbrace{\sum_{k\in S}\tfrac{n_k}{m_t}\,v_k}_{\text{self-normalized (Hajek ratio, }O(1/m)\text{ bias)}}\!,\qquad
\underbrace{g_{\mathrm{HT}}(S)=\tfrac{K}{m}\sum_{k\in S}\tfrac{n_k}{n}\,v_k}_{\mathbb{E}_S[g_{\mathrm{HT}}]=\bar v\ \text{(exactly unbiased)}}.
$$
With $p_k=m/K$, inverse-probability weighting gives $\mathbb{E}_S[g_{\mathrm{HT}}]=\sum_k p_k\frac{1}{p_k}\frac{n_k}{n}v_k=\bar v$. Size-proportional sampling ($p_k\propto n_k$) makes the plain mean over $S$ unbiased too.

## Why this matters
Fixes the $O(1/m)$ bias of the erratum-corrected average (paper node 10) under client imbalance; appears as M.2, Eqs. (a)–(b) of 05-improvements.tex and the §5–§6 ratio-vs-IPW distinction in 02-math-deep-dive.md.

## Code
The aligned runnable demo lives at [`../code/02-unbiased-aggregation-estimators.py`](../code/02-unbiased-aggregation-estimators.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Unbiased aggregation under client imbalance (FedAvg, 05-improvements.tex M.2)
K=100 clients, m=C*K=10, dim=8, 200,000 Monte-Carlo draws
Target vbar = sum_k (n_k/n) v_k; we report bias ||E[agg]-vbar||
```

## Try it yourself
- Exercise 1: Reduce the size spread in `make_population` toward balanced ($n_k\equiv$ const) and watch the self-normalized bias collapse to the Monte-Carlo floor.
- Exercise 2: Add the size-proportional sampling fix ($p_k\propto n_k$, then plain mean) and confirm it is unbiased too.

## Further reading
- McMahan et al. 2017, *Communication-Efficient Learning of Deep Networks*, arXiv:1602.05629 (Algorithm 1, footnotes 2 & 4).
- 05-improvements.tex, section M.2; 02-math-deep-dive.md, §5–§6.


In [ ]:
"""Unbiased Aggregation Estimators -- concept 02-unbiased-aggregation-estimators
of the improvements learning map.

FedAvg's erratum-corrected average sum_{k in S} (n_k/m_t) v_k is a Hajek ratio
estimator (biased under client imbalance); the Horvitz-Thompson form
(K/m) sum_{k in S} (n_k/n) v_k is exactly unbiased. This Monte-Carlo witness
prints each estimator's bias norm: self-normalized >> 0 while HT ~ 0.
Runnable code analog of concepts/02-unbiased-aggregation-estimators.md.
"""
import numpy as np

np.random.seed(0)
K = 100              # number of clients
C = 0.1              # client fraction -> m = C*K clients sampled per round
M = max(1, int(round(C * K)))   # = 10
D = 8                # dimension of each client's returned vector v_k
N_DRAWS = 200_000    # Monte-Carlo uniform size-M subsets


def make_population(imbalanced):
    """Return (sizes n_k of shape (K,), vectors v_k of shape (K, D)).
    v_k is correlated with n_k so size-induced bias is visible (worst case)."""
    if imbalanced:
        sizes = np.round(np.geomspace(10.0, 1000.0, K)).astype(np.int64)
    else:
        sizes = np.full(K, 100, dtype=np.int64)
    s = (sizes - sizes.min()) / (sizes.max() - sizes.min() + 1e-12)   # in [0,1]
    base = np.random.normal(0.0, 1.0, size=(K, D))
    axis = np.random.normal(0.0, 1.0, size=D)
    v = base + 3.0 * np.outer(s, axis)        # large clients shifted along axis
    return sizes, v


def true_target(sizes, v):
    """vbar = sum_k (n_k/n) v_k -- the full size-weighted mean (the estimand)."""
    return (sizes / sizes.sum()) @ v


def mc_uniform(sizes, v):
    """E[self-normalized] and E[Horvitz-Thompson] over uniform size-M subsets."""
    n, p = sizes.sum(), M / K
    acc_sn = np.zeros(D)
    acc_ht = np.zeros(D)
    done = 0
    while done < N_DRAWS:
        b = min(20_000, N_DRAWS - done)
        keys = np.random.random((b, K))
        S = np.argpartition(keys, M, axis=1)[:, :M]        # (b, M) uniform subsets
        nk, vk = sizes[S], v[S]                            # (b, M), (b, M, D)
        mt = nk.sum(axis=1, keepdims=True)                 # RANDOM normalizer
        acc_sn += np.einsum("bm,bmd->bd", nk / mt, vk).sum(axis=0)         # self-norm
        acc_ht += np.einsum("bm,bmd->bd", (nk / n) / p, vk).sum(axis=0)    # HT, fixed p
        done += b
    return acc_sn / N_DRAWS, acc_ht / N_DRAWS


def main():
    print("Unbiased aggregation under client imbalance (FedAvg, 05-improvements.tex M.2)")
    print(f"K={K} clients, m=C*K={M}, dim={D}, {N_DRAWS:,} Monte-Carlo draws")
    print("Target vbar = sum_k (n_k/n) v_k; we report bias ||E[agg]-vbar||\n")

    sz_i, v_i = make_population(imbalanced=True)
    vbar_i = true_target(sz_i, v_i)
    sn_i, ht_i = mc_uniform(sz_i, v_i)
    bias_sn = float(np.linalg.norm(sn_i - vbar_i))
    bias_ht = float(np.linalg.norm(ht_i - vbar_i))

    sz_b, v_b = make_population(imbalanced=False)
    floor = float(np.linalg.norm(mc_uniform(sz_b, v_b)[0] - true_target(sz_b, v_b)))

    print(f"IMBALANCED  self-norm n_k/m_t (FedAvg)   bias = {bias_sn:.3e}   BIASED")
    print(f"IMBALANCED  Horvitz-Thompson (K/m)(n_k/n) bias = {bias_ht:.3e}   ~0 (unbiased)")
    print(f"BALANCED    self-norm n_k/m_t (sanity)    bias = {floor:.3e}   ~0 (MC floor)")
    print(f"\nSelf-normalized bias is {bias_sn / max(bias_ht, 1e-300):.0f}x the HT bias.")

    assert bias_sn > 10 * floor, "self-norm should be clearly biased under imbalance"
    assert bias_ht <= 5 * floor, "HT should sit at the MC floor (unbiased)"
    print("PASS -- Horvitz-Thompson removes the O(1/m) self-normalized bias.")


if __name__ == "__main__":
    main()

# --- run the witness ---
main()


# Adaptive Local-Work Schedule
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `03-adaptive-local-work-schedule`
**Graph:** `improvements`
**Prerequisites:** [paper:07-local-update-count](../../paper/concepts/07-local-update-count.md), [paper:08-heterogeneity-gap-covariance](../../paper/concepts/08-heterogeneity-gap-covariance.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg fixes the local epochs $E$ for the whole run. But a large $E$ lets each client over-optimize its own (non-IID) data and drift out of the shared basin, so progress plateaus. The fix: treat $E$ like a learning rate — start large for fast early progress, then step-decay it toward FedSGD-like single steps ($E\to1$) so late-round clients stop overshooting.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $E_t$ | "E-sub-t" | Local epochs used in round $t$ (now scheduled, not fixed) <!-- TODO add to foundations --> |
| $E_{\min},E_{\max}$ | "E-min, E-max" | Floor / starting value of the schedule <!-- TODO add to foundations --> |
| $\tau$ | "tau" | Halving period: $E_t$ halves every $\tau$ rounds <!-- TODO add to foundations --> |
| $t$ | "t" | Communication-round index <!-- TODO add to foundations --> |
| $u_k=En_k/B$ | "u-sub-k" | Local SGD updates client $k$ runs per round <!-- TODO add to foundations --> |
| $E$ | "E" | Local epochs per client per round <!-- TODO add to foundations --> |
| $B$ | "B" | Local minibatch size ($B=\infty$ ⇒ full batch) <!-- TODO add to foundations --> |

## Formal definition
$$ E_t \;=\; \max\!\Big(E_{\min},\ \big\lfloor E_{\max}\,2^{-\lfloor t/\tau\rfloor}\big\rfloor\Big),\qquad t=0,1,\dots,R-1. $$
Large $E$ early; geometric step-decay toward FedSGD-like steps ($E_t\to E_{\min}=1$) late. The per-round local work $u_k=E_t n_k/B$ inherits the same decay.

## Why this matters
Dodges the large-$E$ plateau that paper node 08's heterogeneity drift $\Delta=\eta^2\mathrm{Cov}_k(H_k,g_k)$ predicts (02-math-deep-dive.md §3, Eq. 3.5) — the failure McMahan et al. flag (Fig. 3) but never fix. Implements 05-improvements.tex E.1 with no extra client state or communication.

## Code
The aligned runnable demo lives at [`../code/03-adaptive-local-work-schedule.py`](../code/03-adaptive-local-work-schedule.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
E_t schedule  E_t = max(1, floor(64 * 2^-floor(t/12))):
  t=0:E=64 t=12:E=32 t=24:E=16 t=36:E=8 t=48:E=4
Final test accuracy over R=60 rounds, extreme non-IID FedAvg:
```

## Try it yourself
- Exercise 1: Set `SEP` larger (well-separated blobs). The drift vanishes and fixed-large $E$ stops plateauing — confirming the plateau is a heterogeneity effect, not a generic one.
- Exercise 2: Replace the $2^{-\lfloor t/\tau\rfloor}$ decay with a linear or cosine schedule on $E_t$; compare final accuracy against the geometric one.

## Further reading
- McMahan et al. 2017, *Communication-Efficient Learning of Deep Networks from Decentralized Data* (arXiv:1602.05629), §3 and Fig. 3.


In [ ]:
"""Adaptive Local-Work Schedule -- concept 03-adaptive-local-work-schedule of the
improvements learning map. Schedule the per-round local epochs E_t like a learning
rate (large early, decay toward FedSGD late) to dodge the large-E client-drift
plateau the FedAvg paper flags but never fixes (05-improvements.tex E.1).
Runnable code analog of concepts/03-adaptive-local-work-schedule.md.
"""
import numpy as np

np.random.seed(0)
np.seterr(over="ignore", divide="ignore", invalid="ignore")

D, C_CLS, K, C, LR, SEP = 12, 6, 40, 0.2, 4.0, 0.25  # small toy; SEP<1 => overlap
R, E_MIN, E_MAX, TAU = 60, 1, 64, 12                  # round budget + schedule knobs


def schedule(t):
    """E_t = max(E_min, floor(E_max * 2^(-floor(t/tau))))  (05-improvements.tex E.1)."""
    return max(E_MIN, int(E_MAX * 2.0 ** (-(t // TAU))))


def make_data(n, rng, centers):
    y = rng.integers(0, C_CLS, size=n)
    X = centers[y] + rng.normal(0, 1.0, size=(n, D)) * np.geomspace(1, 0.05, D)
    return X, y


def softmax(z):
    z = np.clip(z - z.max(1, keepdims=True), -60, 60)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def grad(W, X, y):
    P = softmax(X @ W)
    Y = np.zeros_like(P)
    Y[np.arange(len(y)), y] = 1.0
    return X.T @ (P - Y) / len(y)


def acc(W, X, y):
    return float((np.argmax(X @ W, 1) == y).mean())


def client_step(W, X, y, E):  # full-batch local SGD, E epochs from shared W
    W = W.copy()
    for _ in range(E):
        W -= LR * grad(W, X, y)
    return W


def fed_run(E_of_t, parts, Xtr, ytr, Xte, yte):
    rng = np.random.default_rng(1)
    W = np.zeros((D, C_CLS))                       # shared init each run
    m = max(1, int(round(C * K)))
    sizes = np.array([len(p) for p in parts])
    for t in range(R):
        S = rng.choice(K, m, replace=False)
        Wn = np.zeros_like(W)
        for k in S:                                # corrected n_k/m_t average
            Wk = client_step(W, Xtr[parts[k]], ytr[parts[k]], E_of_t(t))
            Wn += sizes[k] / sizes[S].sum() * Wk
        W = Wn
    return acc(W, Xte, yte)


def main():
    rng = np.random.default_rng(0)
    centers = rng.normal(0, SEP, (C_CLS, D)) * np.geomspace(1, 0.05, D)
    Xtr, ytr = make_data(2400, rng, centers)
    Xte, yte = make_data(1200, rng, centers)
    order = np.argsort(ytr, kind="stable")         # extreme non-IID: 1 class/client
    shards = np.array_split(order, K)
    parts = [shards[i] for i in rng.permutation(K)]

    print("E_t schedule  E_t = max(%d, floor(%d * 2^-floor(t/%d))):" % (E_MIN, E_MAX, TAU))
    print("  " + " ".join("t=%d:E=%d" % (t, schedule(t)) for t in range(0, R, TAU)))
    decayed = fed_run(schedule, parts, Xtr, ytr, Xte, yte)
    print("Final test accuracy over R=%d rounds, extreme non-IID FedAvg:" % R)
    best_fixed, best_E = -1.0, None
    for E in (1, 4, 16, 64):
        a = fed_run(lambda t, E=E: E, parts, Xtr, ytr, Xte, yte)
        tag = "  (fixed large-E plateau)" if E == E_MAX else ""
        print("  fixed E=%-3d  acc=%.2f%%%s" % (E, 100 * a, tag))
        if a > best_fixed:
            best_fixed, best_E = a, E
    print("  DECAYED %d->%d acc=%.2f%%  (best fixed=E=%d:%.2f%%)"
          % (E_MAX, E_MIN, 100 * decayed, best_E, 100 * best_fixed))
    print("VERDICT: decayed schedule beats best fixed E: %s (+%.2f pts)"
          % (decayed >= best_fixed - 1e-9, 100 * (decayed - best_fixed)))


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Compute-Accounting Pareto
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `04-compute-accounting-pareto`
**Graph:** `improvements`
**Prerequisites:** [paper:07-local-update-count](../../paper/concepts/07-local-update-count.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg reports only *rounds of communication*, justified by the unmeasured premise that on-device computation is "essentially free." But each round each client runs $u_k=En_k/B$ local SGD updates, and total on-device work scales as $R\cdot C\cdot K\cdot u_k$. If you plot rounds **and** total compute together, high-$u$ configs that win on the rounds axis can be *Pareto-dominated* on the compute axis. This node turns rounds-alone reporting into a rounds-vs-compute trade.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $u_k=En_k/B$ | "u-sub-k" | Local SGD updates client $k$ runs per round <!-- TODO add to foundations --> |
| $E$ | "E" | Local epochs per client per round <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $B$ | "B" | Local minibatch size ($B=\infty\Rightarrow$ full batch, $u_k=1$) <!-- TODO add to foundations --> |
| $R$ | "R" | Rounds of communication to hit the target <!-- TODO add to foundations --> |
| $C$ | "C" | Fraction of clients selected per round <!-- TODO add to foundations --> |
| $K$ | "K" | Total number of clients <!-- TODO add to foundations --> |

## Formal definition
$$
u_k=\frac{E\,n_k}{B},\qquad
\text{Compute}_{\text{total}}\ \propto\ R\cdot C\cdot K\cdot u_k
\;=\; R\cdot(CK)\cdot\frac{E\,n_k}{B}.
$$
Report the **Pareto frontier** $\{(R,\ \text{Compute}_{\text{total}})\}$ over $(E,B)$ configs, not $R$ alone: a config is dominated if some other config has both fewer rounds and less total compute.

## Why this matters
Tests FedAvg's unquantified "computation is free" premise (Gaps Flagged, 02-math-deep-dive.md; 05-improvements.tex E.2, design): high-$u$ configurations such as $E{=}20,B{=}10$ ($u_k$ up to 1200/round) look best on rounds but may be Pareto-dominated on total compute, revealing a genuine knee.

## Code
The aligned runnable demo lives at [`../code/04-compute-accounting-pareto.py`](../code/04-compute-accounting-pareto.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.
**Expected output preview:**
```
Compute-Accounting Pareto for FedAvg (toy rounds-to-target, n_k=600, C*K=10)
config            u_k |  rounds R | total compute (R*CK*u_k) | Pareto?
E=1,  B=inf         1 |       40  |               4.00e+02   | on-frontier
```

## Try it yourself
- Exercise 1: Change the hardcoded rounds so a high-$u$ config also wins on compute; watch the frontier shrink to one point.
- Exercise 2: Add a per-round communication cost (one model up+down) and plot a 3-way rounds/compute/comm trade.

## Further reading
- McMahan et al. 2017, "Communication-Efficient Learning of Deep Networks from Decentralized Data" (arXiv:1602.05629), Table 2 and the "computation is free" remark.


In [ ]:
"""Compute-Accounting Pareto -- concept 04-compute-accounting-pareto of the improvements learning map.

FedAvg reports rounds-to-target alone, assuming on-device compute is "free". Here we
pair each (E,B) config's rounds R with its total local-update FLOP-proxy R*(C*K)*u_k,
revealing the rounds-vs-compute knee where a high-u config is Pareto-dominated.
Runnable code analog of concepts/04-compute-accounting-pareto.py.
"""
import numpy as np

# Toy fixed setup (matches the paper's MNIST family): n_k=600 examples/client,
# C*K = 10 clients selected per round. u_k = E*n_k/B, with B=inf => u_k=1.
N_K = 600          # examples per client
CK = 10            # C*K selected clients per round
INF = float("inf")


def updates_per_round(E, B):
    """u_k = E*n_k/B local SGD updates; B=inf means one full-batch step => u_k=1."""
    if B == INF:
        return float(E)            # E epochs, 1 batch each => u_k = E (and E=1 => FedSGD)
    return E * N_K / B


def main():
    np.random.seed(0)
    # Hardcoded VERIFIED vanilla-FedAvg rounds-to-target for a few (E,B) configs
    # (toy numbers in the spirit of Table 2: more local work -> fewer rounds).
    configs = [
        # (label,           E,   B,    rounds R)
        ("E=1,  B=inf",      1, INF,   40),   # FedSGD endpoint: cheap/round, many rounds
        ("E=5,  B=inf",      5, INF,   23),
        ("E=20, B=inf",     20, INF,   13),
        ("E=5,  B=10",       5,  10,   11),   # u_k=300: cheaper compute AND fewer rounds
        ("E=20, B=10",      20,  10,   12),   # u_k=1200: diminishing returns -> DOMINATED
    ]

    rows = []
    for label, E, B, R in configs:
        u = updates_per_round(E, B)
        compute = R * CK * u           # total on-device local-update FLOP-proxy
        rows.append((label, u, R, compute))

    # Pareto test on (rounds, compute): minimize BOTH. A config is dominated if some
    # other config has rounds <= and compute <= with at least one strictly less.
    def dominated(i):
        Ri, Ci = rows[i][2], rows[i][3]
        for j, (_, _, Rj, Cj) in enumerate(rows):
            if j == i:
                continue
            if Rj <= Ri and Cj <= Ci and (Rj < Ri or Cj < Ci):
                return True
        return False

    print("Compute-Accounting Pareto for FedAvg (toy rounds-to-target, n_k=600, C*K=10)")
    print("config            u_k |  rounds R | total compute (R*CK*u_k) | Pareto?")
    for i, (label, u, R, compute) in enumerate(rows):
        tag = "DOMINATED  " if dominated(i) else "on-frontier"
        print("{:<16}{:>5.0f} | {:>8d}  | {:>22.2e}   | {}".format(label, u, R, compute, tag))

    # Identify the knee: the on-frontier config minimizing rounds, and the one
    # minimizing compute -- the trade lives between them.
    frontier = [r for i, r in enumerate(rows) if not dominated(i)]
    fewest_rounds = min(frontier, key=lambda r: r[2])
    least_compute = min(frontier, key=lambda r: r[3])
    print("---")
    print("Fewest-rounds frontier config : {:<12} ({} rounds, compute {:.2e})".format(
        fewest_rounds[0], fewest_rounds[2], fewest_rounds[3]))
    print("Least-compute frontier config : {:<12} ({} rounds, compute {:.2e})".format(
        least_compute[0], least_compute[2], least_compute[3]))
    dom = [r[0] for i, r in enumerate(rows) if dominated(i)]
    print("Pareto-DOMINATED configs (lose on BOTH axes vs another): {}".format(
        dom if dom else "none"))
    print("LESSON: reporting rounds alone hides that the rounds-winner is not the "
          "compute-winner -- the knee is the trade FedAvg never measured (E.2).")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Permutation-Aligned Averaging
**Level:** `extension`
**Concept ID:** `05-permutation-aligned-averaging`
**Graph:** `improvements`
**Prerequisites:** [paper:12-shared-init-permutation-barrier](../../paper/concepts/12-shared-init-permutation-barrier.md), [linear assignment problem](https://github.com/pleyva2004/math-foundations/blob/main/concepts/linear-assignment-problem.md)
**Used by:** downstream nodes

## Plain-English intro
A one-hidden-layer net is unchanged if you relabel its hidden units, so two nets trained from independent inits land in *different* relabelings of the same solution and naive averaging blends unrelated units (the loss barrier of paper node 12). Before averaging, we instead solve a linear assignment problem for the permutation $P$ that best matches one net's hidden units to the other's, then average the aligned weights. This is Git Re-Basin weight-matching, and it relaxes FedAvg's shared-initialization requirement.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $W_1$ | "W-one" | Server model's first-layer (hidden) weight matrix <!-- TODO add to foundations --> |
| $W_1'$ | "W-one-prime" | Client model's first-layer weight matrix <!-- TODO add to foundations --> |
| $P$ | "P" | A permutation (matrix) relabeling the $H$ hidden units <!-- TODO add to foundations --> |
| $\Pi_H$ | "cap-Pi-sub-H" | Set of all $H\times H$ permutation matrices <!-- TODO add to foundations --> |
| $H$ | "H" | Number of hidden units <!-- TODO add to foundations --> |
| $\lVert\cdot\rVert_F$ | "Frobenius norm" | Entrywise Euclidean norm of a matrix <!-- TODO add to foundations --> |
| $w,\,w'$ | "w, w-prime" | Full parameter vectors of the server / client models <!-- TODO add to foundations --> |

## Formal definition
$$
P^\star=\arg\min_{P\in\Pi_H}\big\lVert W_1-W_1'P^\top\big\rVert_F^2
       =\arg\max_{P\in\Pi_H}\sum_{i=1}^{H}\big\langle (W_1)_{:,i},\,(W_1')_{:,P(i)}\big\rangle,
$$
since the column norms are permutation-invariant, only the cross-term depends on $P$ — a **linear assignment problem**. The aligned client $w'\mapsto P^\star(w')$ is a function-preserving symmetry ($W_1'\mapsto W_1'P^\top,\ b_1'\mapsto Pb_1',\ W_2'\mapsto PW_2'$), and the server averages $\tfrac12 w+\tfrac12 P^\star(w')$.

## Why this matters
Relaxes FedAvg's shared-init requirement (paper node 12) by aligning independently-trained client models into a common basin via Git Re-Basin weight-matching; this is proposal T.1 of `05-improvements.tex`.

## Code
The aligned runnable demo lives at [`../code/05-permutation-aligned-averaging.py`](../code/05-permutation-aligned-averaging.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Permutation-aligned averaging witness (Git Re-Basin weight-matching)
H = 6 hidden units; client is a relabeling Q = [0, 5, 3, 2, 1, 4]
Hidden-layer Frobenius distance ||W1 - W1'||_F  unmatched = 7.2038
```

## Try it yourself
- Exercise 1: Drop the noise (`0.05 -> 0.0`). The matched distance and `max |f_client - f_aligned|` both go to ~0, and the recovered permutation is exactly $Q^{-1}$.
- Exercise 2: Increase the noise (`0.05 -> 0.5`) until greedy matching mis-pairs a column; check that the matched distance still beats unmatched but the function is no longer preserved.

## Further reading
- Ainsworth, Hayase, Srinivasa, "Git Re-Basin: Merging Models modulo Permutation Symmetries," ICLR 2023.


In [ ]:
"""Permutation-Aligned Averaging -- concept 05-permutation-aligned-averaging of the improvements learning map.

Two one-hidden-layer MLPs from independent inits sit in different elements of the
hidden-unit permutation orbit; weight-matching recovers the permutation P that aligns
their hidden units so averaging blends matched (not unrelated) units.
Runnable code analog of concepts/05-permutation-aligned-averaging.py.md.
"""
import numpy as np


def frob(A, B):
    """Frobenius distance ||A - B||_F."""
    return float(np.linalg.norm(A - B))


def greedy_match(W1, W1p):
    """Greedy column assignment maximizing sum_i <W1[:,i], W1p[:,P(i)]>.

    Returns perm so that W1p[:, perm] best aligns column-by-column with W1.
    Equivalent target to argmin_P ||W1 - W1p P^T||_F^2 (cross-term is the only
    P-dependent part since the column norms are permutation-invariant).
    """
    H = W1.shape[1]
    score = W1.T @ W1p                       # score[i, j] = <W1[:,i], W1p[:,j]>
    perm = -np.ones(H, dtype=int)
    used = np.zeros(H, dtype=bool)
    order = np.dstack(np.unravel_index(np.argsort(-score, axis=None), score.shape))[0]
    for i, j in order:                       # descending inner products
        if perm[i] == -1 and not used[j]:
            perm[i] = j
            used[j] = True
    return perm


def main():
    np.random.seed(0)
    D, H = 4, 6                              # input dim, hidden units

    # Server model w = (W1, W2); independent-init client model w' = (W1p, W2p).
    W1 = np.random.randn(D, H)
    W2 = np.random.randn(H, 1)
    # Build the client as a TRUE permutation+noise of the server: a relabeling Q
    # of hidden units plus small training noise -> same orbit, different element.
    Q = np.random.permutation(H)
    W1p = W1[:, Q] + 0.05 * np.random.randn(D, H)
    W2p = W2[Q, :] + 0.05 * np.random.randn(H, 1)

    print("Permutation-aligned averaging witness (Git Re-Basin weight-matching)")
    print("H =", H, "hidden units; client is a relabeling Q =", Q.tolist())

    # (1) Recover the alignment permutation P from W1 vs W1'.
    perm = greedy_match(W1, W1p)             # perm[i] = which client col matches server col i
    inv = np.argsort(perm)                   # apply P consistently to client weights

    d_unmatched = frob(W1, W1p)
    d_matched = frob(W1, W1p[:, perm])
    print("Hidden-layer Frobenius distance ||W1 - W1'||_F  unmatched = %.4f" % d_unmatched)
    print("Hidden-layer Frobenius distance ||W1 - P(W1')||_F matched  = %.4f" % d_matched)
    print("matched < unmatched :", d_matched < d_unmatched,
          "(recovered perm == Q^-1 :", bool(np.array_equal(perm, np.argsort(Q))), ")")

    # (2) Function-preserving: applying P consistently (W1'->cols, W2'->rows) leaves
    # the client's function unchanged.  Compare outputs on random inputs.
    X = np.random.randn(20, D)
    relu = lambda Z: np.maximum(Z, 0.0)
    f_client = relu(X @ W1p) @ W2p
    W1p_al = W1p[:, perm]                    # permute hidden columns of layer 1
    W2p_al = W2p[perm, :]                    # permute matching rows of layer 2
    f_aligned = relu(X @ W1p_al) @ W2p_al
    max_fn_err = float(np.max(np.abs(f_client - f_aligned)))
    print("max |f_client - f_aligned| = %.2e  (permutation preserves the function)" % max_fn_err)

    # (3) Averaging: naive vs aligned, distance of the average to the server model.
    avg_naive = 0.5 * W1 + 0.5 * W1p
    avg_align = 0.5 * W1 + 0.5 * W1p_al
    print("dist(server, naive avg)   = %.4f" % frob(W1, avg_naive))
    print("dist(server, aligned avg) = %.4f" % frob(W1, avg_align))
    print("VERDICT:", "PASS" if frob(W1, avg_align) < frob(W1, avg_naive) and max_fn_err < 1e-9
          else "FAIL", "-- alignment yields a closer (compatible) average.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Federated LoRA Communication
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `06-federated-lora-communication`
**Graph:** `improvements`
**Prerequisites:** [paper:04-fedsgd-gradient-descent](../../paper/concepts/04-fedsgd-gradient-descent.md), [low-rank factorization](https://github.com/pleyva2004/math-foundations/blob/main/concepts/low-rank-factorization.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg's whole premise is that sending the model each round costs more than the on-device compute. For a big model that hurts: a full layer $W$ is $d\times d$ parameters. LoRA freezes $W$ (shared once) and trains only a thin low-rank adapter $\Delta W = BA$; running FedAvg over *just the adapter* drops per-round communication from $\Theta(d^2)$ to $\Theta(2rd)$, a fraction of $2r/d$.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $W\in\mathbb{R}^{d\times d}$ | "cap-W in R d-by-d" | Frozen base weight matrix (never communicated) <!-- TODO add to foundations --> |
| $\Delta W$ | "delta-W" | Trained low-rank update added to $W$ <!-- TODO add to foundations --> |
| $A\in\mathbb{R}^{r\times d}$ | "cap-A" | Down-projection factor of the adapter <!-- TODO add to foundations --> |
| $B\in\mathbb{R}^{d\times r}$ | "cap-B" | Up-projection factor of the adapter <!-- TODO add to foundations --> |
| $r$ | "r" | Adapter rank, $r\ll d$ <!-- TODO add to foundations --> |
| $d$ | "d" | Layer width / parameter dimension <!-- TODO add to foundations --> |
| $w$ | "w" | The FedAvg model — here only the adapter $(A,B)$ <!-- TODO add to foundations --> |

## Formal definition
$$
W\in\mathbb{R}^{d\times d}\ \text{frozen},\qquad
\Delta W = BA,\quad B\in\mathbb{R}^{d\times r},\ A\in\mathbb{R}^{r\times d},\ \operatorname{rank}(\Delta W)\le r,
$$
$$
\text{FedAvg over } w=(A,B):\qquad
\underbrace{\lvert A\rvert+\lvert B\rvert = 2rd}_{\text{per-round comms}}\ \;\text{vs.}\;\ \underbrace{\lvert W\rvert = d^2}_{\text{full model}},
\qquad \text{ratio}=\frac{2rd}{d^2}=\frac{2r}{d}.
$$

## Why this matters
Makes FedAvg's compute-for-communication trade dramatic for large models: per-round comms become $\Theta(2rd)\ll\Theta(d^2)$, the modern realization of the communication-reduction program of konecny2016 (05-improvements.tex T.2, design).

## Code
The aligned runnable demo lives at [`../code/06-federated-lora-communication.py`](../code/06-federated-lora-communication.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Federated LoRA communication (05-improvements.tex T.2)
Freeze base W in R^{d x d}; FedAvg only adapter Delta W = B A,
  A in R^{r x d}, B in R^{d x r}  ->  per-round comms Theta(2rd) vs Theta(d^2).
```

## Try it yourself
- Exercise 1: Add a non-square base $W\in\mathbb{R}^{d_\text{out}\times d_\text{in}}$ and rederive the ratio (now $r(d_\text{in}+d_\text{out})/(d_\text{in}d_\text{out})$).
- Exercise 2: Pick a target budget (say 1% comms) and solve $2r/d$ for the largest admissible rank $r$ at each $d$ in the table.

## Further reading
- 05-improvements.tex, section T.2; sandbox `real_fedlora.py` (FedAvg over LoRA adapters, ~4.3% of params).
- Konecny et al., *Federated Learning: Strategies for Improving Communication Efficiency*, arXiv:1610.05492.


In [ ]:
"""Federated LoRA Communication -- concept 06-federated-lora-communication
of the improvements learning map.

Freezing a base weight W (d x d) and FedAvg-ing only a low-rank adapter
Delta W = B A (B: d x r, A: r x d) cuts per-round communication from Theta(d^2)
to Theta(2 r d); the ratio 2r/d shrinks as the base model grows. This witness
tabulates base vs adapter param counts and the communication ratio for several
(d, r), including a ~4.3% case matching the sandbox real_fedlora.py.

Runnable code analog of concepts/06-federated-lora-communication.md.
"""

import numpy as np

np.random.seed(0)


def counts(d, r):
    """Base params (d^2), adapter params (2*r*d), and comm ratio (2r/d)."""
    base = d * d
    adapter = 2 * r * d
    ratio = adapter / base          # == 2r/d, the per-round comm fraction
    return base, adapter, ratio


def main():
    print("Federated LoRA communication (05-improvements.tex T.2)")
    print("Freeze base W in R^{d x d}; FedAvg only adapter Delta W = B A,")
    print("  A in R^{r x d}, B in R^{d x r}  ->  per-round comms Theta(2rd) vs Theta(d^2).\n")

    # (d, r) pairs spanning toy -> large; the last targets ~4.3% (sandbox figure).
    configs = [(64, 8), (256, 8), (1024, 16), (4096, 16), (1024, 22)]

    header = "{:>7} {:>5} | {:>14} {:>14} | {:>10} {:>9}".format(
        "d", "r", "base d^2", "adapter 2rd", "ratio 2r/d", "savings"
    )
    print(header)
    print("-" * len(header))
    for d, r in configs:
        base, adapter, ratio = counts(d, r)
        # cross-check the closed form 2r/d against the explicit param counts
        assert abs(ratio - (2 * r / d)) < 1e-12
        assert adapter < base, "LoRA only saves when 2rd < d^2, i.e. 2r < d"
        print("{:>7} {:>5} | {:>14,} {:>14,} | {:>9.2f}% {:>8.1f}x".format(
            d, r, base, adapter, 100.0 * ratio, base / adapter
        ))

    # Numerical witness that B A truly reconstructs a rank-r matrix of shape d x d
    # (so the adapter, not the base, carries the trained update).
    d, r = 1024, 22
    A = np.random.normal(0.0, 0.02, size=(r, d))
    B = np.random.normal(0.0, 0.02, size=(d, r))
    # errstate guards a benign numpy-2.0 + Accelerate BLAS SIMD quirk on Apple
    # Silicon that raises spurious FP flags here; the product is finite & valid.
    with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
        dW = B @ A
    assert np.isfinite(dW).all()
    rank = int(np.linalg.matrix_rank(dW))
    base, adapter, ratio = counts(d, r)
    print("\nWitness (d={}, r={}): Delta W = B@A has shape {} and rank {} (<= r).".format(
        d, r, dW.shape, rank
    ))
    print("  base d^2 = {:,} params; adapter 2rd = {:,} params communicated/round.".format(
        base, adapter
    ))
    print("  per-round communication ratio 2r/d = {:.1f}%  ({:.0f}x less than the base).".format(
        100.0 * ratio, base / adapter
    ))
    assert rank <= r, "B@A cannot exceed rank r"
    print("PASS -- adapter is rank <= r and costs only 2r/d of the base each round.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()
